In [1]:
# ============================================================
# İETT SENTINEL — HÜCRE 1: Drive Bağlantısı + Kurulum
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

import os
BASE_PATH = '/content/drive/MyDrive/iett_sentinel/data/'

# Klasör yapısını oluştur
klasorler = [
    'iett_sentinel/data/Dimensions',
    'iett_sentinel/data/DATATHON_DENETIM',
    'iett_sentinel/data/DATATHON_SEFER',
    'iett_sentinel/data/DATATHON_YOLCULUK',
    'iett_sentinel/data/API_DATA',
    'iett_sentinel/outputs/figures',
    'iett_sentinel/outputs/models',
    'iett_sentinel/outputs/reports',
]
for k in klasorler:
    os.makedirs(f'/content/drive/MyDrive/{k}', exist_ok=True)

print("✅ Drive bağlandı!")
print("✅ Klasör yapısı hazır!")
print(f"📁 Base path: {BASE_PATH}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Drive bağlandı!
✅ Klasör yapısı hazır!
📁 Base path: /content/drive/MyDrive/iett_sentinel/data/


In [2]:
# ============================================================
# İETT SENTINEL — HÜCRE 2: KÜTÜPHANELERo + VERİ YÜKLEME
# ============================================================

import warnings
warnings.filterwarnings('ignore')

!pip install -q mlxtend plotly kaleido folium

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import requests, json, os, gc

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)
print("✅ Kütüphaneler hazır\n")

# ── Yükleme Fonksiyonları ─────────────────────────────────
def yukle(dosya, usecols=None, nrows=None):
    yol = BASE_PATH + dosya
    if not os.path.exists(yol):
        print(f"❌ BULUNAMADI: {dosya}")
        return None
    df = pd.read_csv(yol, usecols=usecols, nrows=nrows,
                     low_memory=False, on_bad_lines='skip')
    mb = os.path.getsize(yol) / 1e6
    print(f"✅ {dosya:<45} {df.shape[0]:>10,} satır  {mb:>7.1f} MB")
    return df

def yukle_klasor(klasor, usecols=None):
    yol = BASE_PATH + klasor
    if not os.path.exists(yol):
        print(f"❌ BULUNAMADI: {klasor}")
        return None
    dfs = []
    for f in sorted(os.listdir(yol)):
        if f.endswith('.csv'):
            df = pd.read_csv(os.path.join(yol, f), usecols=usecols,
                             low_memory=False, on_bad_lines='skip')
            df['_kaynak'] = f
            dfs.append(df)
            print(f"  📄 {f:<40} {df.shape[0]:>10,} satır")
    if dfs:
        combined = pd.concat(dfs, ignore_index=True)
        print(f"  ➕ {klasor} TOPLAM: {combined.shape[0]:,} satır\n")
        return combined
    return None

# ── Dimension Tabloları ───────────────────────────────────
print("=" * 60)
print("📦 DIMENSION TABLOLARI")
print("=" * 60)
d_arac       = yukle('Dimensions/D_ARAC.csv')
d_hat        = yukle('Dimensions/D_HAT.csv')
d_garaj      = yukle('Dimensions/D_GARAJ.csv')
d_personel   = yukle('Dimensions/D_PERSONEL.csv')
d_tarih      = yukle('Dimensions/D_TARIH.csv')
d_durak      = yukle('Dimensions/D_DURAK.csv')
d_guzergah   = yukle('Dimensions/D_GUZERGAH.csv')
d_kilavuz    = yukle('Dimensions/D_KILAVUZ.csv')
d_kaza_kil   = yukle('Dimensions/D_KAZA_KILAVUZ.csv')
d_kaza_kat   = yukle('Dimensions/D_KAZA_KATEGORI.csv')
d_denetimtip = yukle('Dimensions/D_DENETIMTIP.csv')
d_ulasimturu = yukle('Dimensions/D_ULASIMTURU.csv')

# ── Fact Tabloları ────────────────────────────────────────
print("\n" + "=" * 60)
print("📦 FACT TABLOLARI")
print("=" * 60)
f_ariza       = yukle('F_ARIZA.csv')
f_kaza        = yukle('F_KAZA.csv')
f_kaza_analiz = yukle('F_KAZA_ANALIZ.csv')

# ── Denetim (büyük — sadece gerekli kolonlar) ─────────────
print("\n" + "=" * 60)
print("📦 DENETİM (yüklenince otomatik okur)")
print("=" * 60)
DENETIM_COLS = [
    'FID','ID','SIDKAYITTARIHI','SIDPERSONELSOFOR',
    'SIDDENETIMTIP','SIDVARLIK','DENETIMDURUM',
    'CEVAPDURUM','VARLIKTIP','VARLIKKODU','VARLIKOPERATOR'
]
f_denetim = yukle_klasor('DATATHON_DENETIM', usecols=DENETIM_COLS)

# ── Sefer (yüklendiyse) ───────────────────────────────────
print("=" * 60)
print("📦 SEFER (yüklenince otomatik okur)")
print("=" * 60)
SEFER_COLS = [
    'FID','TASKID','SIDOPERASYONGUNU','SIDGUZERGAH',
    'SIDGOREVDURUMU','SIDGOREVTIPI','SIDDURUMDEGISIKLIK',
    'SIDARAC','SIDHAT','SIDPERSONEL','SIDARACGARAJ',
    'BASLANGICZAMANI','BITISZAMANI','PLANLANANBASLANGICZAMANI',
    'TOPLAMKM','GUZERGAHUZUNLUK','SEFERSIRASI'
]
f_sefer = yukle_klasor('DATATHON_SEFER', usecols=SEFER_COLS)

# ── Yolculuk (yüklendiyse) ────────────────────────────────
print("=" * 60)
print("📦 YOLCULUK (yüklenince otomatik okur)")
print("=" * 60)
YOLCULUK_COLS = [
    'FID','GECISZAMANI','SIDTARIH','SIDARAC',
    'SIDHAT','SIDDURAK','SIDULASIMTURU',
    'SIDONCEKIHAT','DGECISTURU'
]
f_yolculuk = yukle_klasor('DATATHON_YOLCULUK', usecols=YOLCULUK_COLS)

# ── Özet ─────────────────────────────────────────────────
print("\n" + "=" * 60)
print("📊 YÜKLEME ÖZETİ")
print("=" * 60)

tablolar = {
    'f_ariza': f_ariza, 'f_kaza': f_kaza,
    'f_kaza_analiz': f_kaza_analiz, 'f_denetim': f_denetim,
    'f_sefer': f_sefer, 'f_yolculuk': f_yolculuk,
    'd_arac': d_arac, 'd_hat': d_hat,
    'd_garaj': d_garaj, 'd_tarih': d_tarih,
    'd_durak': d_durak, 'd_guzergah': d_guzergah,
}

toplam = 0
for isim, df in tablolar.items():
    if df is not None:
        print(f"  ✅ {isim:<20} {df.shape[0]:>10,} satır × {df.shape[1]:>3} kolon")
        toplam += df.shape[0]
    else:
        print(f"  ⏳ {isim:<20} henüz yüklenmedi")

print(f"\n  📈 TOPLAM: {toplam:,} satır işlenmeye hazır")
print("\n🎯 Veri yükleme tamamlandı!")

✅ Kütüphaneler hazır

📦 DIMENSION TABLOLARI
✅ Dimensions/D_ARAC.csv                              8,684 satır      1.2 MB
✅ Dimensions/D_HAT.csv                               1,625 satır      0.1 MB
✅ Dimensions/D_GARAJ.csv                               116 satır      0.0 MB
✅ Dimensions/D_PERSONEL.csv                         59,679 satır      2.3 MB
✅ Dimensions/D_TARIH.csv                            55,153 satır      2.4 MB
✅ Dimensions/D_DURAK.csv                            16,851 satır      1.8 MB
✅ Dimensions/D_GUZERGAH.csv                        481,671 satır     60.8 MB
✅ Dimensions/D_KILAVUZ.csv                             657 satır      0.0 MB
✅ Dimensions/D_KAZA_KILAVUZ.csv                      1,173 satır      0.1 MB
✅ Dimensions/D_KAZA_KATEGORI.csv                        10 satır      0.0 MB
✅ Dimensions/D_DENETIMTIP.csv                           49 satır      0.0 MB
✅ Dimensions/D_ULASIMTURU.csv                            4 satır      0.0 MB

📦 FACT TABLOLARI
✅ F_ARIZA.csv 

In [28]:
# ============================================================
# HÜCRE 2B — EKSİK SEFER DOSYALARI (Q2 2024 + Q1 2025)
# Mevcut f_sefer'e eklenir
# ============================================================

import os, gc

SEFER_COLS = [
    'FID','TASKID','SIDOPERASYONGUNU','SIDGUZERGAH',
    'SIDGOREVDURUMU','SIDGOREVTIPI','SIDDURUMDEGISIKLIK',
    'SIDARAC','SIDHAT','SIDPERSONEL','SIDARACGARAJ',
    'BASLANGICZAMANI','BITISZAMANI','PLANLANANBASLANGICZAMANI',
    'TOPLAMKM','GUZERGAHUZUNLUK','SEFERSIRASI'
]

SEFER_PATH = BASE_PATH + 'DATATHON_SEFER/'

print("=" * 60)
print("📦 SEFER DOSYALARI KONTROL & YÜKLEME")
print("=" * 60)

# Mevcut f_sefer'deki kaynak dosyaları kontrol et
mevcut_dosyalar = set()
if '_kaynak' in f_sefer.columns:
    mevcut_dosyalar = set(f_sefer['_kaynak'].unique())
print(f"  Mevcut f_sefer: {len(f_sefer):,} satır")
print(f"  Mevcut dosyalar: {mevcut_dosyalar}")

# Tüm sefer dosyalarını listele
tum_dosyalar = sorted([f for f in os.listdir(SEFER_PATH) if f.endswith('.csv')])
print(f"  Drive'daki tüm sefer dosyaları: {tum_dosyalar}")

# Eksik olanları yükle
yeni_dfs = []
for dosya in tum_dosyalar:
    if dosya in mevcut_dosyalar:
        print(f"  ⏭️  {dosya} zaten yüklü, atlanıyor")
        continue
    yol = SEFER_PATH + dosya
    mb  = os.path.getsize(yol) / 1e6
    print(f"\n  ⬇️  {dosya} ({mb:.0f} MB) yükleniyor...")
    df = pd.read_csv(yol, usecols=SEFER_COLS,
                     low_memory=False, on_bad_lines='skip')
    df['_kaynak'] = dosya
    print(f"  ✅ {dosya}: {df.shape[0]:,} satır")
    yeni_dfs.append(df)

if yeni_dfs:
    f_sefer_ek = pd.concat(yeni_dfs, ignore_index=True)
    f_sefer    = pd.concat([f_sefer, f_sefer_ek], ignore_index=True)
    del f_sefer_ek, yeni_dfs
    gc.collect()
    print(f"\n✅ f_sefer GÜNCELLENDİ: {len(f_sefer):,} satır")
    print(f"   Kaynak dağılımı:")
    print(f_sefer['_kaynak'].value_counts().to_string())
else:
    print("\n✅ Tüm dosyalar zaten yüklüydü, f_sefer değişmedi.")
    print(f"   Mevcut f_sefer: {len(f_sefer):,} satır")

📦 SEFER DOSYALARI KONTROL & YÜKLEME
  Mevcut f_sefer: 10,085,460 satır
  Mevcut dosyalar: {'SEFER2024_Q1.csv', 'SEFER2025_Q1.csv'}
  Drive'daki tüm sefer dosyaları: ['SEFER2024_Q1.csv', 'SEFER2024_Q2.csv', 'SEFER2025_Q1.csv']
  ⏭️  SEFER2024_Q1.csv zaten yüklü, atlanıyor

  ⬇️  SEFER2024_Q2.csv (1824 MB) yükleniyor...
  ✅ SEFER2024_Q2.csv: 4,937,211 satır
  ⏭️  SEFER2025_Q1.csv zaten yüklü, atlanıyor

✅ f_sefer GÜNCELLENDİ: 15,022,671 satır
   Kaynak dağılımı:
_kaynak
SEFER2025_Q1.csv    5073846
SEFER2024_Q1.csv    5011614
SEFER2024_Q2.csv    4937211


In [24]:
# ============================================================
# İETT SENTINEL — HÜCRE 3: EDA + VERİ TEMİZLEME
# ============================================================

print("=" * 60)
print("🔍 1. VERİ TANIMLAMA — GENEL BAKIŞ")
print("=" * 60)

def veri_tanimi(df, isim):
    print(f"\n{'─'*50}")
    print(f"📊 {isim}")
    print(f"{'─'*50}")
    print(f"Boyut      : {df.shape[0]:,} satır × {df.shape[1]} kolon")
    print(f"Bellek     : {df.memory_usage(deep=True).sum()/1e6:.1f} MB")
    print(f"Veri Tipleri:\n{df.dtypes.value_counts().to_string()}")
    eksik = df.isnull().sum()
    eksik_oran = (eksik / len(df) * 100).round(1)
    eksik_df = pd.DataFrame({
        'Eksik Sayı': eksik,
        'Eksik %': eksik_oran
    }).query('`Eksik Sayı` > 0').sort_values('Eksik %', ascending=False)
    if len(eksik_df) > 0:
        print(f"\nEksik Veri ({len(eksik_df)} kolon etkilenmiş):")
        print(eksik_df.head(10).to_string())
    else:
        print("\n✅ Eksik veri yok")
    duplik = df.duplicated().sum()
    print(f"\nTekrarlayan satır: {duplik:,}")
    return eksik_df

eksik_ariza = veri_tanimi(f_ariza,  "F_ARIZA")
eksik_kaza  = veri_tanimi(f_kaza,   "F_KAZA")
eksik_arac  = veri_tanimi(d_arac,   "D_ARAC")
eksik_hat   = veri_tanimi(d_hat,    "D_HAT")
eksik_garaj = veri_tanimi(d_garaj,  "D_GARAJ")

# ============================================================
print("\n\n" + "=" * 60)
print("📈 2. F_ARIZA — DERİN ANALİZ")
print("=" * 60)

f_ariza['OLAYTARIHI']              = pd.to_datetime(f_ariza['OLAYTARIHI'], errors='coerce')
f_ariza['KAYITSAATI']              = pd.to_datetime(f_ariza['KAYITSAATI'], errors='coerce')
f_ariza['MUDEHALEBASLANGICTARIHI'] = pd.to_datetime(f_ariza['MUDEHALEBASLANGICTARIHI'], errors='coerce')
f_ariza['MUDEHALESONU']            = pd.to_datetime(f_ariza['MUDEHALESONU'], errors='coerce')
f_ariza['BILDIRIMTARIHI']          = pd.to_datetime(f_ariza['BILDIRIMTARIHI'], errors='coerce')
f_ariza['ZAYISEFERSAYISI']         = pd.to_numeric(f_ariza['ZAYISEFERSAYISI'], errors='coerce').fillna(0)
f_ariza['CEKICITALEP']             = pd.to_numeric(f_ariza['CEKICITALEP'], errors='coerce').fillna(0)
f_ariza['ENLEM']                   = pd.to_numeric(f_ariza['ENLEM'], errors='coerce')
f_ariza['BOYLAM']                  = pd.to_numeric(f_ariza['BOYLAM'], errors='coerce')

f_ariza['YIL']           = f_ariza['OLAYTARIHI'].dt.year
f_ariza['AY']            = f_ariza['OLAYTARIHI'].dt.month
f_ariza['SAAT']          = f_ariza['OLAYTARIHI'].dt.hour
f_ariza['HAFTANIN_GUNU'] = f_ariza['OLAYTARIHI'].dt.day_name()
f_ariza['HAFTASONU']     = f_ariza['OLAYTARIHI'].dt.dayofweek >= 5

f_ariza['MUDAHALE_SURE_DK'] = (
    f_ariza['MUDEHALESONU'] - f_ariza['MUDEHALEBASLANGICTARIHI']
).dt.total_seconds() / 60
f_ariza['MUDAHALE_SURE_DK'] = f_ariza['MUDAHALE_SURE_DK'].clip(0, 600)

f_ariza['BILDIRIM_GECIKME_DK'] = (
    f_ariza['MUDEHALEBASLANGICTARIHI'] - f_ariza['BILDIRIMTARIHI']
).dt.total_seconds() / 60
f_ariza['BILDIRIM_GECIKME_DK'] = f_ariza['BILDIRIM_GECIKME_DK'].clip(0, 300)

istanbul_mask = (
    f_ariza['ENLEM'].between(40.8, 41.5) &
    f_ariza['BOYLAM'].between(28.0, 29.9)
)
f_ariza['KONUM_GECERLI'] = istanbul_mask

print("✅ F_ARIZA tarih kolonları düzeltildi")
print(f"   Tarih aralığı: {f_ariza['OLAYTARIHI'].min()} → {f_ariza['OLAYTARIHI'].max()}")
print(f"   Geçersiz tarih: {f_ariza['OLAYTARIHI'].isna().sum():,} kayıt")
print(f"   Sefer iptali olan arıza: {(f_ariza['ZAYISEFERSAYISI']>0).sum():,} ({(f_ariza['ZAYISEFERSAYISI']>0).mean()*100:.1f}%)")
print(f"   Çekici talep edilen: {f_ariza['CEKICITALEP'].sum():,.0f}")
print(f"   Geçerli konum verisi: {istanbul_mask.sum():,} ({istanbul_mask.mean()*100:.1f}%)")

# ============================================================
print("\n\n" + "=" * 60)
print("📈 3. F_KAZA — DERİN ANALİZ")
print("=" * 60)

f_kaza['KAZASAAT'] = pd.to_datetime(f_kaza['KAZASAAT'], errors='coerce')
f_kaza['ENLEM']    = pd.to_numeric(f_kaza['ENLEM'], errors='coerce')
f_kaza['BOYLAM']   = pd.to_numeric(f_kaza['BOYLAM'], errors='coerce')
f_kaza['HIZSON']   = pd.to_numeric(f_kaza['HIZSON'], errors='coerce')

kaza_kolonlar = ['YARALISURUCU','OLUMSURUCU','OLUMYOLCU','YARALIYOLCU',
                 'OLUMYAYA','YARALIYAYA','OLUMSIVIL','YARALISIVIL']
for k in kaza_kolonlar:
    f_kaza[k] = pd.to_numeric(f_kaza[k], errors='coerce').fillna(0)

f_kaza['TOPLAM_YARALI'] = f_kaza[['YARALISURUCU','YARALIYOLCU','YARALIYAYA','YARALISIVIL']].sum(axis=1)
f_kaza['TOPLAM_OLUM']   = f_kaza[['OLUMSURUCU','OLUMYOLCU','OLUMYAYA','OLUMSIVIL']].sum(axis=1)
f_kaza['CIDDI_KAZA']    = (f_kaza['TOPLAM_OLUM'] > 0) | (f_kaza['TOPLAM_YARALI'] > 2)

f_kaza_zengin = f_kaza.merge(
    d_kaza_kil[d_kaza_kil['KLAVUZUSTID']==125][['SID','ADI']].rename(
        columns={'SID':'SIDHAVADURUMU','ADI':'HAVA_DURUMU'}),
    on='SIDHAVADURUMU', how='left'
).merge(
    d_kaza_kil[d_kaza_kil['KLAVUZUSTID']==133][['SID','ADI']].rename(
        columns={'SID':'SIDYOLDURUMU','ADI':'YOL_DURUMU'}),
    on='SIDYOLDURUMU', how='left'
).merge(
    d_kaza_kil[d_kaza_kil['KLAVUZUSTID']==140][['SID','ADI']].rename(
        columns={'SID':'SIDTRAFIKDURUMU','ADI':'TRAFIK_DURUMU'}),
    on='SIDTRAFIKDURUMU', how='left'
)

print(f"✅ F_KAZA zenginleştirildi")
print(f"   Tarih aralığı  : {f_kaza['KAZASAAT'].min()} → {f_kaza['KAZASAAT'].max()}")
print(f"   Toplam yaralı  : {f_kaza['TOPLAM_YARALI'].sum():.0f}")
print(f"   Toplam ölüm    : {f_kaza['TOPLAM_OLUM'].sum():.0f}")
print(f"   Ciddi kaza     : {f_kaza['CIDDI_KAZA'].sum():,}")
print(f"   Şoför kusurlu  : {f_kaza['SOFORKUSUR'].notna().sum():,}")

# ============================================================
print("\n\n" + "=" * 60)
print("📈 4. D_ARAC — TEMİZLEME")
print("=" * 60)

d_arac['MODELYILI']    = pd.to_numeric(d_arac['MODELYILI'], errors='coerce')
d_arac['KAPASITE']     = pd.to_numeric(d_arac['KAPASITE'], errors='coerce')
d_arac['KOLTUKSAYISI'] = pd.to_numeric(d_arac['KOLTUKSAYISI'], errors='coerce')
d_arac['ARAC_YASI']    = (2025 - d_arac['MODELYILI']).clip(0, 30)

for k in ['MARKA','MODEL','YAKITTURU','ARACTIPI','USTOPERATORADI']:
    if k in d_arac.columns:
        d_arac[k] = d_arac[k].astype(str).str.strip().str.upper()

print(f"✅ D_ARAC temizlendi")
print(f"   Araç yaşı aralığı: {d_arac['ARAC_YASI'].min():.0f} – {d_arac['ARAC_YASI'].max():.0f} yıl")
print(f"   Ortalama araç yaşı: {d_arac['ARAC_YASI'].mean():.1f} yıl")
print(f"   Marka çeşidi: {d_arac['MARKA'].nunique()}")
print(f"   Yakıt türü: {d_arac['YAKITTURU'].value_counts().to_dict()}")
print(f"   Operatör: {d_arac['USTOPERATORADI'].value_counts().to_dict()}")

# ============================================================
print("\n\n" + "=" * 60)
print("📊 5. HIZLI GÖRSELLEŞTİRME")
print("=" * 60)

import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go

aylik = f_ariza.groupby(['YIL','AY']).size().reset_index(name='ariza_sayisi')
aylik['TARIH'] = pd.to_datetime(aylik['YIL'].astype(str) + '-' + aylik['AY'].astype(str) + '-01')
fig1 = px.line(aylik, x='TARIH', y='ariza_sayisi', color='YIL',
               title='📈 Aylık Arıza Trend (2024 vs 2025)',
               template='plotly_dark', markers=True,
               labels={'ariza_sayisi':'Arıza Sayısı','TARIH':'Tarih'})
fig1.show()

saatlik = f_ariza['SAAT'].value_counts().sort_index().reset_index()
saatlik.columns = ['Saat','Arıza Sayısı']
fig2 = px.bar(saatlik, x='Saat', y='Arıza Sayısı',
              title='🕐 Saate Göre Arıza Dağılımı',
              template='plotly_dark', color='Arıza Sayısı',
              color_continuous_scale='Reds')
fig2.show()

top_ariza = f_ariza['ARIZAUSTKODTANIM'].value_counts().head(12).reset_index()
top_ariza.columns = ['Arıza Türü','Adet']
fig3 = px.bar(top_ariza, x='Adet', y='Arıza Türü', orientation='h',
              title='🔧 En Sık 12 Arıza Türü',
              template='plotly_dark', color='Adet',
              color_continuous_scale='Oranges')
fig3.update_layout(height=500)
fig3.show()

zayi_top = f_ariza[f_ariza['ZAYISEFERSAYISI']>0].groupby('ARIZAUSTKODTANIM')[
    'ZAYISEFERSAYISI'].sum().sort_values(ascending=False).head(10).reset_index()
zayi_top.columns = ['Arıza Türü','Toplam Zayi Sefer']
fig4 = px.bar(zayi_top, x='Toplam Zayi Sefer', y='Arıza Türü', orientation='h',
              title='🚫 Sefer İptaline Yol Açan Arıza Türleri',
              template='plotly_dark', color='Toplam Zayi Sefer',
              color_continuous_scale='Reds')
fig4.update_layout(height=450)
fig4.show()

if 'HAVA_DURUMU' in f_kaza_zengin.columns:
    hava = f_kaza_zengin['HAVA_DURUMU'].value_counts().reset_index()
    hava.columns = ['Hava Durumu','Kaza Sayısı']
    fig5 = px.pie(hava, names='Hava Durumu', values='Kaza Sayısı',
                  title='🌤️ Hava Durumuna Göre Kaza Dağılımı',
                  template='plotly_dark')
    fig5.show()

fig6 = px.histogram(d_arac, x='ARAC_YASI', nbins=20,
                    title='🚌 Filodaki Araç Yaşı Dağılımı',
                    template='plotly_dark', color_discrete_sequence=['#3498db'],
                    labels={'ARAC_YASI':'Araç Yaşı (yıl)'})
fig6.show()

# ============================================================
print("\n\n" + "=" * 60)
print("✅ TEMİZLEME ÖZETİ")
print("=" * 60)
print(f"  F_ARIZA  : {len(f_ariza):,} kayıt — tarihler parse edildi, koordinatlar doğrulandı")
print(f"  F_KAZA   : {len(f_kaza_zengin):,} kayıt — hava/yol/trafik durumu eklendi")
print(f"  D_ARAC   : {len(d_arac):,} araç — yaş hesaplandı, metin normalize edildi")
print(f"\n🎯 Veri temiz! Modül 1'e geçmeye hazırız.")

# ============================================================
# TEMİZ VERSİYONLAR — Tüm modüller bu değişkenleri kullanır
# ============================================================
print("\n" + "=" * 60)
print("🧹 TEMİZ VERSİYONLAR OLUŞTURULUYOR")
print("=" * 60)

# ── f_ariza_temiz ─────────────────────────────────────────
f_ariza_temiz = f_ariza[f_ariza['OLAYTARIHI'].notna()].copy()
f_ariza_temiz['YIL']       = f_ariza_temiz['OLAYTARIHI'].dt.year
f_ariza_temiz['AY']        = f_ariza_temiz['OLAYTARIHI'].dt.month
f_ariza_temiz['SAAT']      = f_ariza_temiz['OLAYTARIHI'].dt.hour
f_ariza_temiz['HAFTASONU'] = f_ariza_temiz['OLAYTARIHI'].dt.dayofweek >= 5
f_ariza_temiz['MUDAHALE_SURE_DK'] = (
    f_ariza_temiz['MUDEHALESONU'] - f_ariza_temiz['MUDEHALEBASLANGICTARIHI']
).dt.total_seconds() / 60
f_ariza_temiz['MUDAHALE_SURE_DK'] = f_ariza_temiz['MUDAHALE_SURE_DK'].clip(0, 600)
f_ariza_temiz['ZAYISEFERSAYISI']  = pd.to_numeric(
    f_ariza_temiz['ZAYISEFERSAYISI'], errors='coerce').fillna(0)
f_ariza_temiz['CEKICITALEP']      = pd.to_numeric(
    f_ariza_temiz['CEKICITALEP'], errors='coerce').fillna(0)
print(f"  ✅ f_ariza_temiz : {len(f_ariza_temiz):,} kayıt")

# ── f_kaza_temiz ──────────────────────────────────────────
f_kaza_temiz = f_kaza_zengin.copy()
f_kaza_temiz['KAZA_YIL']   = f_kaza_temiz['KAZASAAT'].dt.year
f_kaza_temiz['KAZA_AY']    = f_kaza_temiz['KAZASAAT'].dt.month
f_kaza_temiz['KAZA_SAATI'] = f_kaza_temiz['KAZASAAT'].dt.hour
f_kaza_temiz['KAZA_GUNU']  = f_kaza_temiz['KAZASAAT'].dt.day_name()
f_kaza_temiz['HAFTASONU']  = f_kaza_temiz['KAZASAAT'].dt.dayofweek >= 5
f_kaza_temiz['ILCE']        = f_kaza_temiz['ILCE'].fillna('BELİRSİZ')
f_kaza_temiz['HAVA_DURUMU'] = f_kaza_temiz['HAVA_DURUMU'].fillna('BELİRSİZ')
f_kaza_temiz['YOL_DURUMU']  = f_kaza_temiz['YOL_DURUMU'].fillna('BELİRSİZ')
f_kaza_temiz['HIZSON']      = pd.to_numeric(f_kaza_temiz['HIZSON'], errors='coerce')

# ── Resmi tatil — D_TARIH 2000'den başlıyor, manuel tanımla ──
_tatil_set = set(pd.to_datetime([
    '2024-01-01','2024-04-10','2024-04-11','2024-04-12','2024-04-13',
    '2024-04-23','2024-05-01','2024-05-19',
    '2024-06-16','2024-06-17','2024-06-18','2024-06-19',
    '2024-07-15','2024-08-30','2024-10-29',
    '2025-01-01','2025-03-30','2025-03-31','2025-04-01','2025-04-02',
    '2025-04-23','2025-05-01','2025-05-19',
    '2025-06-06','2025-06-07','2025-06-08','2025-06-09',
    '2025-07-15','2025-08-30','2025-10-29',
]).date)
f_kaza_temiz['KAZASAAT']   = pd.to_datetime(f_kaza_temiz['KAZASAAT'], errors='coerce')
f_kaza_temiz['RESMITATIL'] = f_kaza_temiz['KAZASAAT'].dt.date.apply(
    lambda x: 1 if x in _tatil_set else 0)

# ── KRİZ_TETIKLEYICI — RESMITATIL eklendikten SONRA hesaplanır ──
f_kaza_temiz['KRİZ_TETIKLEYICI'] = (
    (f_kaza_temiz['TOPLAM_OLUM'] > 0) |
    (f_kaza_temiz['TOPLAM_YARALI'] >= 3) |
    (f_kaza_temiz['HAVA_DURUMU'].isin(['Yagmurlu','Karli','Sisli','Firtina'])) |
    (f_kaza_temiz['RESMITATIL'] == 1)
).astype(int)
print(f"  ✅ f_kaza_temiz  : {len(f_kaza_temiz):,} kaza  |  "
      f"Kriz tetikleyici: {f_kaza_temiz['KRİZ_TETIKLEYICI'].sum():,}")

# ── d_arac_temiz ──────────────────────────────────────────
d_arac_temiz = d_arac.copy()
d_arac_temiz['MODELYILI'] = pd.to_numeric(d_arac_temiz['MODELYILI'], errors='coerce')
d_arac_temiz['ARAC_YASI'] = (2025 - d_arac_temiz['MODELYILI']).clip(0, 30)

def yas_kat(y):
    if pd.isna(y):  return 'Bilinmiyor'
    if y <= 3:      return 'Yeni (0-3)'
    if y <= 7:      return 'Genç (4-7)'
    if y <= 12:     return 'Orta (8-12)'
    return 'Yaşlı (13+)'

d_arac_temiz['YAS_KATEGORI'] = d_arac_temiz['ARAC_YASI'].apply(yas_kat)
for k in ['MARKA','MODEL','YAKITTURU','ARACTIPI','USTOPERATORADI']:
    if k in d_arac_temiz.columns:
        d_arac_temiz[k] = d_arac_temiz[k].astype(str).str.strip().str.upper()
print(f"  ✅ d_arac_temiz  : {len(d_arac_temiz):,} araç  |  "
      f"Yaş kategorileri: {d_arac_temiz['YAS_KATEGORI'].value_counts().to_dict()}")

# ── d_hat_temiz ───────────────────────────────────────────
d_hat_temiz = d_hat.copy()
d_hat_temiz['HATCINSI'] = d_hat_temiz['HATCINSI'].fillna('BİLİNMİYOR')
d_hat_temiz['HATILCE']  = d_hat_temiz['HATILCE'].fillna('BİLİNMİYOR')
print(f"  ✅ d_hat_temiz   : {len(d_hat_temiz):,} hat")

print(f"\n🎯 Tüm temiz versiyonlar hazır!")
print(f"   → f_ariza_temiz, f_kaza_temiz, d_arac_temiz, d_hat_temiz")

🔍 1. VERİ TANIMLAMA — GENEL BAKIŞ

──────────────────────────────────────────────────
📊 F_ARIZA
──────────────────────────────────────────────────
Boyut      : 333,386 satır × 40 kolon
Bellek     : 329.3 MB
Veri Tipleri:
object            12
float64           12
int64              6
datetime64[ns]     5
int32              3
bool               2

Eksik Veri (23 kolon etkilenmiş):
                  Eksik Sayı  Eksik %
ENLEM                 333386   100.00
BOYLAM                333386   100.00
SIDIMDATPERSONEL      318527    95.50
SIDYEDEKARAC          313398    94.00
SIDDEGISENARAC        313398    94.00
YERBILGISI            245993    73.80
SIDYARDIMCIARAC        99621    29.90
FIRMASONLANDIRMA       99717    29.90
SONLANDIRMA            99619    29.90
IMDATARAC              99621    29.90

Tekrarlayan satır: 0

──────────────────────────────────────────────────
📊 F_KAZA
──────────────────────────────────────────────────
Boyut      : 11,941 satır × 36 kolon
Bellek     : 4.7 MB
Veri Tipl



✅ TEMİZLEME ÖZETİ
  F_ARIZA  : 333,386 kayıt — tarihler parse edildi, koordinatlar doğrulandı
  F_KAZA   : 11,941 kayıt — hava/yol/trafik durumu eklendi
  D_ARAC   : 8,684 araç — yaş hesaplandı, metin normalize edildi

🎯 Veri temiz! Modül 1'e geçmeye hazırız.

🧹 TEMİZ VERSİYONLAR OLUŞTURULUYOR
  ✅ f_ariza_temiz : 333,386 kayıt
  ✅ f_kaza_temiz  : 11,941 kaza  |  Kriz tetikleyici: 403
  ✅ d_arac_temiz  : 8,684 araç  |  Yaş kategorileri: {'Orta (8-12)': 3924, 'Bilinmiyor': 1534, 'Yaşlı (13+)': 1329, 'Yeni (0-3)': 1273, 'Genç (4-7)': 624}
  ✅ d_hat_temiz   : 1,625 hat

🎯 Tüm temiz versiyonlar hazır!
   → f_ariza_temiz, f_kaza_temiz, d_arac_temiz, d_hat_temiz


In [4]:
# ============================================================
# İETT SENTINEL —  HÜCRE 4    MODÜL 1: ARIZA RİSK KÜMELEMESİ
# K-Means + PCA ile Araç Bazlı Risk Profilleme
# ============================================================

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import gc

print("=" * 60)
print("🔧 MODÜL 1: ARIZA RİSK KÜMELEMESİ")
print("=" * 60)

# ── ADIM 1: Arıza + Araç Birleştirme ─────────────────────
print("\n📎 ADIM 1: Tablolar birleştiriliyor...")

df_m1 = f_ariza_temiz.merge(
    d_arac_temiz[[
        'SID','MARKA','MODELYILI','YAKITTURU','ARACTIPI',
        'GARAJKODU','USTOPERATORADI','ARAC_YASI','YAS_KATEGORI'
    ]],
    left_on='SIDARAC', right_on='SID', how='left'
).merge(
    d_garaj[['SID','GARAJADI','ILCE']].rename(columns={'ILCE':'GARAJ_ILCE'}),
    left_on='SIDGARAJ', right_on='SID', how='left',
    suffixes=('','_garaj')
).merge(
    d_hat_temiz[['SID','HATADI','HATCINSI','HATILCE']],
    left_on='SIDHAT', right_on='SID', how='left',
    suffixes=('','_hat')
)

print(f"  ✅ Birleştirme tamamlandı: {df_m1.shape[0]:,} kayıt")

# ── ADIM 2: Araç Bazında Feature Engineering ─────────────
print("\n🔨 ADIM 2: Araç bazında özellikler hesaplanıyor...")

arac_features = df_m1.groupby('SIDARAC').agg(
    toplam_ariza        =('FID',               'count'),
    toplam_zayi_sefer   =('ZAYISEFERSAYISI',   'sum'),
    ort_mudahale_sure   =('MUDAHALE_SURE_DK',  'mean'),
    cekici_talep_sayisi =('CEKICITALEP',        'sum'),
    benzersiz_ariza_tip =('ARIZAUSTKODTANIM',  'nunique'),
    gece_ariza_sayisi   =('SAAT',   lambda x: (x >= 22).sum() + (x <= 6).sum()),
    haftasonu_ariza     =('HAFTASONU',          'sum'),
    arac_yasi           =('ARAC_YASI',          'first'),
    marka               =('MARKA',              'first'),
    yakitturu           =('YAKITTURU',           'first'),
    aractipi            =('ARACTIPI',            'first'),
    yas_kategori        =('YAS_KATEGORI',        'first'),
    garaj               =('GARAJKODU',           'first'),
    garaj_ilce          =('GARAJ_ILCE',          'first'),
    ustoperator         =('USTOPERATORADI',      'first'),
    hatcinsi            =('HATCINSI',            'first'),
).reset_index()

# Oran bazlı özellikler
arac_features['GECE_ARİZA_ORANI']     = (
    arac_features['gece_ariza_sayisi'] / arac_features['toplam_ariza']
).fillna(0)
arac_features['HAFTASONU_ARİZA_ORANI'] = (
    arac_features['haftasonu_ariza'] / arac_features['toplam_ariza']
).fillna(0)

# Ham risk skoru
arac_features['HAM_RISK_SKORU'] = (
    arac_features['toplam_ariza']        * 1.0 +
    arac_features['toplam_zayi_sefer']   * 2.5 +
    arac_features['cekici_talep_sayisi'] * 3.0 +
    arac_features['benzersiz_ariza_tip'] * 1.5 +
    arac_features['arac_yasi'].fillna(0) * 0.3
)

print(f"  ✅ {arac_features.shape[0]:,} araç için {arac_features.shape[1]} özellik hazır")

# ── ADIM 3: K-Means Kümeleme ──────────────────────────────
print("\n🤖 ADIM 3: K-Means kümeleme...")

num_cols = [
    'toplam_ariza', 'toplam_zayi_sefer', 'ort_mudahale_sure',
    'cekici_talep_sayisi', 'benzersiz_ariza_tip',
    'GECE_ARİZA_ORANI', 'arac_yasi'
]

X = arac_features[num_cols].fillna(0)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Optimal k — Elbow + Silhouette
inertias, sil_scores = [], []
K_range = range(2, 8)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_scaled, labels))
    print(f"  k={k} → Silhouette: {sil_scores[-1]:.3f}  |  Inertia: {inertias[-1]:.1f}")

optimal_k = list(K_range)[sil_scores.index(max(sil_scores))]
print(f"\n  ✅ Optimal küme sayısı: {optimal_k} (Silhouette: {max(sil_scores):.3f})")

# Final model
kmeans_final = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
arac_features['KUME'] = kmeans_final.fit_predict(X_scaled)

# ── ADIM 4: Kümeleri Risk Etiketle ───────────────────────
print("\n🏷️  ADIM 4: Kümeler etiketleniyor...")

kume_stats = arac_features.groupby('KUME').agg(
    arac_sayisi   =('SIDARAC',           'count'),
    ort_ariza     =('toplam_ariza',       'mean'),
    ort_zayi      =('toplam_zayi_sefer',  'mean'),
    ort_cekici    =('cekici_talep_sayisi','mean'),
    ort_risk      =('HAM_RISK_SKORU',     'mean'),
    ort_yas       =('arac_yasi',          'mean'),
).round(2)

print("\n📊 Küme İstatistikleri:")
print(kume_stats.to_string())

# Risk sırasına göre etiket ata
risk_labels = ['🔴 YÜKSEK RİSK','🟠 ORTA-YÜKSEK','🟡 ORTA RİSK',
               '🟢 DÜŞÜK RİSK','🔵 ÇOK DÜŞÜK','⚪ MİNİMAL']
risk_sirasi  = kume_stats['ort_risk'].rank(ascending=False).astype(int)
risk_etiket  = {idx: risk_labels[rank-1] for idx, rank in risk_sirasi.items()}
arac_features['RİSK_SEVİYESİ'] = arac_features['KUME'].map(risk_etiket)

print(f"\n  ✅ Risk dağılımı:")
print(arac_features['RİSK_SEVİYESİ'].value_counts().to_string())

# ── ADIM 5: Görselleştirmeler ─────────────────────────────
print("\n📊 ADIM 5: Görselleştirmeler...")

renk_map = {
    '🔴 YÜKSEK RİSK':    '#e74c3c',
    '🟠 ORTA-YÜKSEK':    '#e67e22',
    '🟡 ORTA RİSK':      '#f1c40f',
    '🟢 DÜŞÜK RİSK':     '#2ecc71',
    '🔵 ÇOK DÜŞÜK':      '#3498db',
    '⚪ MİNİMAL':         '#95a5a6',
}

# PCA 2D
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)
arac_features['PCA1'] = X_pca[:, 0]
arac_features['PCA2'] = X_pca[:, 1]

print(f"  PCA açıklanan varyans: %{sum(pca.explained_variance_ratio_)*100:.1f}")

fig1 = px.scatter(
    arac_features, x='PCA1', y='PCA2',
    color='RİSK_SEVİYESİ',
    color_discrete_map=renk_map,
    hover_data=['SIDARAC','toplam_ariza','toplam_zayi_sefer',
                'arac_yasi','marka','garaj_ilce'],
    title='🔧 Araç Arıza Risk Kümeleri (PCA)',
    template='plotly_dark', width=900, height=500
)
fig1.update_traces(marker=dict(size=5, opacity=0.7))
fig1.show()

# Risk dağılımı bar
dagilim = arac_features['RİSK_SEVİYESİ'].value_counts().reset_index()
dagilim.columns = ['Risk','Adet']

fig2 = px.bar(
    dagilim, x='Risk', y='Adet',
    color='Risk', color_discrete_map=renk_map,
    title='🚌 Risk Seviyesine Göre Araç Sayısı',
    template='plotly_dark', text='Adet', width=750, height=430
)
fig2.update_traces(textposition='outside')
fig2.show()

# Marka × Risk
top_markalar = arac_features['marka'].value_counts().head(8).index
marka_risk = arac_features[arac_features['marka'].isin(top_markalar)].groupby(
    ['marka','RİSK_SEVİYESİ']).size().reset_index(name='sayi')

fig3 = px.bar(
    marka_risk, x='marka', y='sayi',
    color='RİSK_SEVİYESİ', color_discrete_map=renk_map,
    title='🚌 Marka × Risk Seviyesi',
    barmode='stack', template='plotly_dark',
    width=900, height=480
)
fig3.show()

# Yaş kategorisi × Risk
yas_risk = arac_features.groupby(
    ['yas_kategori','RİSK_SEVİYESİ']).size().reset_index(name='sayi')

fig4 = px.bar(
    yas_risk, x='yas_kategori', y='sayi',
    color='RİSK_SEVİYESİ', color_discrete_map=renk_map,
    title='📅 Araç Yaş Kategorisi × Risk Seviyesi',
    barmode='stack', template='plotly_dark',
    width=800, height=450,
    category_orders={'yas_kategori':['Yeni (0-3)','Genç (4-7)','Orta (8-12)','Yaşlı (13+)']}
)
fig4.show()

# Garaj ilçe × ortalama risk skoru
garaj_risk = arac_features.groupby('garaj_ilce').agg(
    ort_risk  =('HAM_RISK_SKORU','mean'),
    arac_sayi =('SIDARAC','count')
).reset_index().sort_values('ort_risk', ascending=False).head(15)

fig5 = px.bar(
    garaj_risk, x='garaj_ilce', y='ort_risk',
    title='🏭 Garaj İlçesine Göre Ortalama Risk Skoru (Top 15)',
    template='plotly_dark', color='ort_risk',
    color_continuous_scale='RdYlGn_r',
    text='arac_sayi', width=900, height=450,
    labels={'garaj_ilce':'Garaj İlçesi','ort_risk':'Ort. Risk Skoru',
            'arac_sayi':'Araç Sayısı'}
)
fig5.update_traces(texttemplate='%{text} araç', textposition='outside')
fig5.show()

# ── ADIM 6: Bakım Öncelik Listesi ─────────────────────────
print("\n" + "=" * 60)
print("🎯 BAKIM ÖNCELİK LİSTESİ — Yüksek Riskli İlk 20 Araç")
print("=" * 60)

oncelik = arac_features[
    arac_features['RİSK_SEVİYESİ'] == '🔴 YÜKSEK RİSK'
].sort_values('HAM_RISK_SKORU', ascending=False)[[
    'SIDARAC','marka','arac_yasi','yas_kategori',
    'toplam_ariza','toplam_zayi_sefer',
    'cekici_talep_sayisi','ort_mudahale_sure',
    'garaj','garaj_ilce','RİSK_SEVİYESİ'
]].head(20)

print(oncelik.to_string(index=False))

# ── Kaydet ───────────────────────────────────────────────
SAVE = '/content/drive/MyDrive/iett_sentinel/outputs/reports/'
arac_features.to_csv(SAVE + 'modul1_arac_risk_kumeleri.csv', index=False)
oncelik.to_csv(SAVE + 'modul1_bakim_oncelik_listesi.csv', index=False)

# Bellek temizle
del df_m1, X, X_scaled, X_pca
gc.collect()

print(f"\n✅ Sonuçlar kaydedildi → {SAVE}")
print(f"\n📊 MODÜL 1 ÖZET:")
print(f"   Analiz edilen araç    : {len(arac_features):,}")
print(f"   Yüksek riskli araç    : {(arac_features['RİSK_SEVİYESİ']=='🔴 YÜKSEK RİSK').sum():,}")
print(f"   Toplam zayi sefer     : {arac_features['toplam_zayi_sefer'].sum():,.0f}")
print(f"   Ort. müdahale süresi  : {arac_features['ort_mudahale_sure'].mean():.1f} dk")
print(f"\n🎯 Modül 1 tamamlandı! → Modül 2 (Birliktelik Analizi)")


🔧 MODÜL 1: ARIZA RİSK KÜMELEMESİ

📎 ADIM 1: Tablolar birleştiriliyor...
  ✅ Birleştirme tamamlandı: 333,386 kayıt

🔨 ADIM 2: Araç bazında özellikler hesaplanıyor...
  ✅ 6,769 araç için 20 özellik hazır

🤖 ADIM 3: K-Means kümeleme...
  k=2 → Silhouette: 0.393  |  Inertia: 28793.5
  k=3 → Silhouette: 0.321  |  Inertia: 24960.3
  k=4 → Silhouette: 0.313  |  Inertia: 21462.0
  k=5 → Silhouette: 0.324  |  Inertia: 18875.1
  k=6 → Silhouette: 0.294  |  Inertia: 16985.0
  k=7 → Silhouette: 0.307  |  Inertia: 15375.9

  ✅ Optimal küme sayısı: 2 (Silhouette: 0.393)

🏷️  ADIM 4: Kümeler etiketleniyor...

📊 Küme İstatistikleri:
      arac_sayisi  ort_ariza  ort_zayi  ort_cekici  ort_risk  ort_yas
KUME                                                                 
0            3120      85.00     48.88        3.15    244.94    12.33
1            3649      18.66     14.92        0.08     60.18     7.05

  ✅ Risk dağılımı:
RİSK_SEVİYESİ
🟠 ORTA-YÜKSEK    3649
🔴 YÜKSEK RİSK    3120

📊 ADIM 5: Görsel


🎯 BAKIM ÖNCELİK LİSTESİ — Yüksek Riskli İlk 20 Araç
 SIDARAC    marka  arac_yasi yas_kategori  toplam_ariza  toplam_zayi_sefer  cekici_talep_sayisi  ort_mudahale_sure  garaj   garaj_ilce RİSK_SEVİYESİ
 4322.00   KARSAN      12.00  Orta (8-12)           185                187                   12               0.07 G_ISL2 Küçükçekmece 🔴 YÜKSEK RİSK
 5336.00 MERCEDES      19.00  Yaşlı (13+)           177                190                    3               0.09  G_IKT Küçükçekmece 🔴 YÜKSEK RİSK
 3262.00   KARSAN      12.00  Orta (8-12)           195                169                    6               0.19 G_ISL2 Küçükçekmece 🔴 YÜKSEK RİSK
 1727.00   KARSAN      12.00  Orta (8-12)           191                153                    4               0.03 G_ISL2 Küçükçekmece 🔴 YÜKSEK RİSK
 2008.00   KARSAN      12.00  Orta (8-12)           146                159                   12               0.12 G_ISL2 Küçükçekmece 🔴 YÜKSEK RİSK
 1902.00   KARSAN      12.00  Orta (8-12)           1

In [5]:
# ============================================================
# İETT SENTINEL — HÜCRE 5   MODÜL 2: ARIZA BİRLİKTELİK & FAKTÖR
# RAM DOSTU VERSİYON
# ============================================================

import gc
import warnings
warnings.filterwarnings('ignore')

print("=" * 60)
print("🔗 MODÜL 2: ARIZA BİRLİKTELİK & FAKTÖR ANALİZİ")
print("=" * 60)

# ── ADIM 1: Arıza Sepeti — Sadece Top Arızalar ────────────
print("\n📦 ADIM 1: Arıza sepeti hazırlanıyor (RAM tasarruflu)...")

# Önce en sık görülen arıza türlerini bul
top_ariza_turleri = (f_ariza_temiz['ARIZAUSTKODTANIM']
                     .value_counts()
                     .head(20)  # Sadece top 20 arıza türü
                     .index.tolist())

print(f"  ✅ Top 20 arıza türü seçildi:")
for i, a in enumerate(top_ariza_turleri, 1):
    print(f"     {i:>2}. {a}")

# Sadece top arıza türlerini filtrele
f_ariza_top = f_ariza_temiz[
    f_ariza_temiz['ARIZAUSTKODTANIM'].isin(top_ariza_turleri)
].copy()

print(f"\n  ✅ Filtrelenmiş kayıt: {len(f_ariza_top):,}")

# Araç bazında arıza listesi
ariza_sepet = (f_ariza_top
               .groupby('SIDARAC')['ARIZAUSTKODTANIM']
               .apply(lambda x: list(x.unique()))
               .reset_index())
ariza_sepet.columns = ['SIDARAC', 'ARIZA_LISTESI']
ariza_sepet = ariza_sepet[
    ariza_sepet['ARIZA_LISTESI'].apply(len) >= 2
]

print(f"  ✅ Sepet oluşturuldu: {len(ariza_sepet):,} araç")

# Bellek temizle
del f_ariza_top
gc.collect()

# ── ADIM 2: Manuel Birliktelik (Apriori yerine) ───────────
print("\n🔍 ADIM 2: Birliktelik analizi (manuel, hafif)...")

from itertools import combinations

# Her ikili kombinasyonun kaç araçta birlikte göründüğünü say
combo_sayac = {}
tekil_sayac = {}
toplam_arac = len(ariza_sepet)

for _, row in ariza_sepet.iterrows():
    ariza_listesi = row['ARIZA_LISTESI']
    # Tekil sayımlar
    for a in ariza_listesi:
        tekil_sayac[a] = tekil_sayac.get(a, 0) + 1
    # İkili kombinasyonlar
    for a, b in combinations(sorted(ariza_listesi), 2):
        key = (a, b)
        combo_sayac[key] = combo_sayac.get(key, 0) + 1

# Birliktelik kuralları hesapla
kurallar = []
min_support = 0.03  # %3

for (a, b), combo_n in combo_sayac.items():
    support = combo_n / toplam_arac
    if support >= min_support:
        # A → B
        if tekil_sayac.get(a, 0) > 0:
            conf_ab = combo_n / tekil_sayac[a]
            lift_ab = conf_ab / (tekil_sayac.get(b,1) / toplam_arac)
            kurallar.append({
                'oncul': a, 'sonuc': b,
                'support': round(support, 4),
                'confidence': round(conf_ab, 4),
                'lift': round(lift_ab, 3),
                'birlikte_gorulme': combo_n
            })
        # B → A
        if tekil_sayac.get(b, 0) > 0:
            conf_ba = combo_n / tekil_sayac[b]
            lift_ba = conf_ba / (tekil_sayac.get(a,1) / toplam_arac)
            kurallar.append({
                'oncul': b, 'sonuc': a,
                'support': round(support, 4),
                'confidence': round(conf_ba, 4),
                'lift': round(lift_ba, 3),
                'birlikte_gorulme': combo_n
            })

rules_df = pd.DataFrame(kurallar).sort_values('lift', ascending=False)
print(f"  ✅ {len(rules_df)} birliktelik kuralı bulundu")

# En güçlü kurallar
print("\n🏆 En Güçlü 10 Birliktelik Kuralı:")
print("-" * 65)
for _, row in rules_df.head(10).iterrows():
    print(f"  {row['oncul'][:30]}")
    print(f"  → {row['sonuc'][:30]}")
    print(f"     Güven: %{row['confidence']*100:.1f}  "
          f"Lift: {row['lift']:.2f}  "
          f"Destek: %{row['support']*100:.1f}  "
          f"({row['birlikte_gorulme']} araç)")
    print()

del ariza_sepet, combo_sayac, tekil_sayac
gc.collect()

# ── ADIM 3: Sefer İptali Zinciri ──────────────────────────
print("\n⛓️  ADIM 3: Sefer iptali zinciri...")

zayi_ozet = (f_ariza_temiz[f_ariza_temiz['ZAYISEFERSAYISI'] > 0]
             .groupby('ARIZAUSTKODTANIM')
             .agg(
                 ariza_sayisi      =('FID',             'count'),
                 toplam_zayi_sefer =('ZAYISEFERSAYISI', 'sum'),
                 ort_zayi_sefer    =('ZAYISEFERSAYISI', 'mean'),
                 cekici_oran       =('CEKICITALEP',     'mean'),
             )
             .round(2)
             .sort_values('toplam_zayi_sefer', ascending=False)
             .head(15))

print("\n🚫 Sefer İptaline En Çok Yol Açan 15 Arıza:")
print(zayi_ozet.to_string())

# ── ADIM 4: Faktör Analizi ────────────────────────────────
print("\n\n📊 ADIM 4: Faktör Analizi...")

# Araç yaşı korelasyonu
corr_cols = ['arac_yasi','toplam_ariza',
             'toplam_zayi_sefer','cekici_talep_sayisi']
corr_data = arac_features[corr_cols].dropna()

print("\n  Araç Yaşı Korelasyonları:")
print(f"  → Yaş × Arıza      : r = {corr_data['arac_yasi'].corr(corr_data['toplam_ariza']):.3f}")
print(f"  → Yaş × Zayi Sefer : r = {corr_data['arac_yasi'].corr(corr_data['toplam_zayi_sefer']):.3f}")
print(f"  → Yaş × Çekici     : r = {corr_data['arac_yasi'].corr(corr_data['cekici_talep_sayisi']):.3f}")

# Marka analizi
print("\n  Marka Bazında Ortalama Arıza:")
marka_ariza = (arac_features.groupby('marka')
               .agg(arac_sayisi =('SIDARAC',          'count'),
                    ort_ariza   =('toplam_ariza',       'mean'),
                    ort_zayi    =('toplam_zayi_sefer',  'mean'))
               .round(1)
               .sort_values('ort_ariza', ascending=False))
print(marka_ariza.to_string())

# Yakıt türü analizi
print("\n  Yakıt Türü Bazında Arıza:")
yakit_ariza = (arac_features.groupby('yakitturu')
               .agg(arac_sayisi =('SIDARAC',         'count'),
                    ort_ariza   =('toplam_ariza',      'mean'),
                    ort_zayi    =('toplam_zayi_sefer', 'mean'))
               .round(1)
               .sort_values('ort_ariza', ascending=False))
print(yakit_ariza.to_string())

# Operatör analizi
print("\n  Operatör Bazında Arıza:")
op_ariza = (arac_features.groupby('ustoperator')
            .agg(arac_sayisi =('SIDARAC',         'count'),
                 ort_ariza   =('toplam_ariza',      'mean'),
                 ort_zayi    =('toplam_zayi_sefer', 'mean'))
            .round(1)
            .sort_values('ort_ariza', ascending=False))
print(op_ariza.to_string())

# ── ADIM 5: Görselleştirme ────────────────────────────────
print("\n📊 ADIM 5: Görselleştirmeler...")

import plotly.express as px

# Grafik 1: Birliktelik kuralları scatter
if len(rules_df) > 0:
    fig1 = px.scatter(
        rules_df.head(30),
        x='support', y='confidence',
        size='lift', color='lift',
        color_continuous_scale='RdYlGn',
        hover_data=['oncul','sonuc','lift','birlikte_gorulme'],
        title='🔗 Birliktelik Kuralları — Support vs Confidence',
        template='plotly_dark', width=850, height=500,
        labels={'support':'Destek','confidence':'Güven'}
    )
    fig1.show()

# Grafik 2: Sefer iptali bar
fig2 = px.bar(
    zayi_ozet.reset_index(),
    x='toplam_zayi_sefer', y='ARIZAUSTKODTANIM',
    orientation='h',
    color='ort_zayi_sefer', color_continuous_scale='Reds',
    title='🚫 Sefer İptaline En Çok Yol Açan Arıza Türleri',
    template='plotly_dark', width=900, height=500,
    labels={'ARIZAUSTKODTANIM': 'Arıza Türü',
            'toplam_zayi_sefer': 'Toplam Zayi Sefer'}
)
fig2.show()

# Grafik 3: Araç yaşı × arıza scatter
fig3 = px.scatter(
    arac_features.sample(min(1500, len(arac_features))),
    x='arac_yasi', y='toplam_ariza',
    color='RİSK_SEVİYESİ',
    color_discrete_map=renk_map,
    title='📅 Araç Yaşı × Arıza Sayısı',
    template='plotly_dark', width=850, height=500,
    labels={'arac_yasi':'Araç Yaşı','toplam_ariza':'Arıza Sayısı'}
)
fig3.show()

# Grafik 4: Marka × ortalama arıza
fig4 = px.bar(
    marka_ariza.reset_index(),
    x='marka', y='ort_ariza',
    color='ort_zayi', color_continuous_scale='Reds',
    title='🏭 Marka Bazında Ortalama Arıza',
    template='plotly_dark', width=850, height=450,
    labels={'marka':'Marka','ort_ariza':'Ort. Arıza Sayısı'}
)
fig4.show()

# ── Kaydet ───────────────────────────────────────────────
SAVE = '/content/drive/MyDrive/iett_sentinel/outputs/reports/'
rules_df.to_csv(SAVE + 'modul2_birliktelik_kurallari.csv', index=False)
zayi_ozet.to_csv(SAVE + 'modul2_zayi_sefer_analizi.csv')

gc.collect()

print(f"\n✅ Sonuçlar kaydedildi → {SAVE}")
print(f"\n📊 MODÜL 2 ÖZET:")
print(f"   Birliktelik kuralı  : {len(rules_df)}")
print(f"   En yüksek lift      : {rules_df['lift'].max():.2f}")
print(f"   En yüksek güven     : {rules_df['confidence'].max()*100:.1f}%")
print(f"   Sefer iptal analizi : {len(zayi_ozet)} arıza türü")
print(f"\n🎯 Modül 2 tamamlandı! → Modül 4 (Kriz Zaman Serisi)")

🔗 MODÜL 2: ARIZA BİRLİKTELİK & FAKTÖR ANALİZİ

📦 ADIM 1: Arıza sepeti hazırlanıyor (RAM tasarruflu)...
  ✅ Top 20 arıza türü seçildi:
      1. SOĞUTMA SİSTEMİ ARIZASI
      2. KAPI ARIZALARI
      3. ELEKTRİK SİSTEMİ ARIZALARI
      4. MOTOR ARIZALARI
      5. KLİMA SİSTEMİ ARIZALARI
      6. KAROSER ARIZALARI
      7. FREN ŞİKAYETLERİ
      8. OTOMATİK ŞANZIMAN ARIZALARI
      9. SÜSPANSİYON SİSTEMİ ARIZALARI
     10. AKBİL ARIZALARI
     11. Destek
     12. YAKIT ve ENJEKSİYON ARIZALARI
     13. ISITMA SİSTEMİ
     14. KAMERA SİSTEMİ ARIZALARI
     15. BASINÇLI HAVA DONANIMI ARIZALARI
     16. KAYIŞ KASNAK ARIZALARI
     17. BASINÇLI MOTOR YAĞI SİSTEMİ
     18. LASTİK ARIZALARI
     19. BASINÇLI HAVA HATTI ARIZASI
     20. BASINÇLI YAĞ HATTI ARIZASI

  ✅ Filtrelenmiş kayıt: 225,643
  ✅ Sepet oluşturuldu: 3,675 araç

🔍 ADIM 2: Birliktelik analizi (manuel, hafif)...
  ✅ 380 birliktelik kuralı bulundu

🏆 En Güçlü 10 Birliktelik Kuralı:
---------------------------------------------------


✅ Sonuçlar kaydedildi → /content/drive/MyDrive/iett_sentinel/outputs/reports/

📊 MODÜL 2 ÖZET:
   Birliktelik kuralı  : 380
   En yüksek lift      : 1.49
   En yüksek güven     : 99.3%
   Sefer iptal analizi : 15 arıza türü

🎯 Modül 2 tamamlandı! → Modül 4 (Kriz Zaman Serisi)


In [6]:
# ============================================================
# İETT SENTINEL — HÜCRE 6: IBB OPEN DATA API VERİ ÇEKME
# ============================================================

import requests, json, os, zipfile, io, gc
import warnings
warnings.filterwarnings('ignore')

API_PATH = BASE_PATH + 'API_DATA/'
GTFS_PATH = API_PATH + 'GTFS/'
os.makedirs(API_PATH, exist_ok=True)
os.makedirs(GTFS_PATH, exist_ok=True)

CKAN_BASE = "https://data.ibb.gov.tr/api/3/action"

print("=" * 60)
print("🌐 IBB OPEN DATA API — VERİ ÇEKME")
print("=" * 60)

# ── Yardımcı Fonksiyonlar ─────────────────────────────────
def kaynak_url_getir(dataset_id):
    try:
        r = requests.get(f"{CKAN_BASE}/package_show",
                         params={'id': dataset_id}, timeout=15)
        return r.json()['result']['resources']
    except Exception as e:
        print(f"  ❌ {e}")
        return []

def indir_chunk(url, kayit_yolu, isim, chunk_mb=5):
    """Büyük dosyaları chunk chunk indir"""
    try:
        print(f"  ⬇️  {isim} indiriliyor...")
        r = requests.get(url, stream=True, timeout=120)
        if r.status_code == 200:
            toplam = 0
            with open(kayit_yolu, 'wb') as f:
                for chunk in r.iter_content(
                        chunk_size=chunk_mb * 1024 * 1024):
                    if chunk:
                        f.write(chunk)
                        toplam += len(chunk)
                        print(f"    → {toplam/1e6:.1f} MB...", end='\r')
            mb = os.path.getsize(kayit_yolu) / 1e6
            print(f"\n  ✅ {isim}: {mb:.1f} MB kaydedildi")
            return True
        else:
            print(f"  ⚠️  HTTP {r.status_code}")
            return False
    except Exception as e:
        print(f"  ❌ {e}")
        return False

def zip_indir_ac(url, hedef_klasor, isim):
    try:
        print(f"  ⬇️  {isim} (ZIP) indiriliyor...")
        r = requests.get(url, timeout=120)
        if r.status_code == 200:
            z = zipfile.ZipFile(io.BytesIO(r.content))
            z.extractall(hedef_klasor)
            dosyalar = z.namelist()
            print(f"  ✅ {isim}: {len(dosyalar)} dosya açıldı → {dosyalar}")
            return True
        else:
            print(f"  ⚠️  HTTP {r.status_code}")
            return False
    except Exception as e:
        print(f"  ❌ {e}")
        return False


# ── 1. İETT Otobüs Durakları (GeoJSON) ───────────────────
print("\n📍 1. İETT Otobüs Durakları...")
durak_yolu = API_PATH + 'iett_otobus_duraklari.geojson'

if not os.path.exists(durak_yolu):
    kaynaklar = kaynak_url_getir('iett-otobus-duraklari-verisi')
    for k in kaynaklar:
        fmt = k.get('format','').upper()
        url = k.get('url','')
        if 'GEOJSON' in fmt or 'geojson' in url.lower():
            indir_chunk(url, durak_yolu, 'Otobüs Durakları')
            break
else:
    mb = os.path.getsize(durak_yolu) / 1e6
    print(f"  ✅ Zaten mevcut: {mb:.1f} MB")


# ── 2. İETT Hat Güzergahları (GeoJSON) ───────────────────
print("\n📍 2. İETT Hat Güzergahları...")
hat_yolu = API_PATH + 'iett_hat_guzergahlari.geojson'

if not os.path.exists(hat_yolu):
    kaynaklar = kaynak_url_getir('iett-hat-guzergahlari')
    for k in kaynaklar:
        fmt = k.get('format','').upper()
        url = k.get('url','')
        if 'GEOJSON' in fmt or 'geojson' in url.lower():
            print(f"  URL: {url[:80]}")
            indir_chunk(url, hat_yolu, 'Hat Güzergahları')
            break
    else:
        print("  ⚠️  GeoJSON kaynağı bulunamadı — atlanıyor")
        print("  💡 Durak koordinatları ile devam edilecek")
else:
    mb = os.path.getsize(hat_yolu) / 1e6
    print(f"  ✅ Zaten mevcut: {mb:.1f} MB")


# ── 3. GTFS Verisi ────────────────────────────────────────
print("\n📍 3. GTFS Verisi...")
kaynaklar = kaynak_url_getir('iett-gtfs-verisi')

print(f"  Mevcut kaynaklar:")
for k in kaynaklar:
    fmt  = k.get('format','').upper()
    isim = k.get('name','')
    url  = k.get('url','')
    print(f"  {fmt:8} | {isim:30} | {url[:55]}")

for k in kaynaklar:
    fmt = k.get('format','').upper()
    url = k.get('url','')
    isim = k.get('name','gtfs')

    if 'ZIP' in fmt or '.zip' in url.lower():
        zip_indir_ac(url, GTFS_PATH, 'GTFS ZIP')
        break
    elif 'CSV' in fmt:
        dosya_adi = isim.replace(' ','_').lower() + '.txt'
        hedef = GTFS_PATH + dosya_adi
        if not os.path.exists(hedef):
            indir_chunk(url, hedef, f'GTFS {isim}')


# ── 4. Otobüs Durak GeoJSON → DataFrame ──────────────────
print("\n📍 4. Durak GeoJSON → DataFrame dönüşümü...")

if os.path.exists(durak_yolu):
    with open(durak_yolu, 'r', encoding='utf-8') as f:
        durak_geojson = json.load(f)

    durak_rows = []
    for feat in durak_geojson['features']:
        props  = feat.get('properties', {})
        geom   = feat.get('geometry', {})
        coords = geom.get('coordinates', [None, None])
        durak_rows.append({
            'DURAK_ID'   : props.get('ID'),
            'DURAK_ADI'  : props.get('ADI'),
            'DURAK_KODU' : props.get('DURAK_KODU'),
            'DURUM'      : props.get('DURUMU'),
            'DURAK_TIPI' : props.get('DURAK_TIPI'),
            'YON_BILGISI': props.get('YON_BILGISI'),
            'BOYLAM'     : coords[0] if coords else None,
            'ENLEM'      : coords[1] if coords else None,
        })

    api_duraklar = pd.DataFrame(durak_rows)
    api_duraklar = api_duraklar.dropna(subset=['BOYLAM','ENLEM'])

    # İstanbul sınırları filtresi
    api_duraklar = api_duraklar[
        api_duraklar['ENLEM'].between(40.8, 41.5) &
        api_duraklar['BOYLAM'].between(28.0, 29.9)
    ]

    print(f"  ✅ Toplam durak      : {len(api_duraklar):,}")
    print(f"  ✅ Enlem aralığı     : "
          f"{api_duraklar['ENLEM'].min():.4f} → "
          f"{api_duraklar['ENLEM'].max():.4f}")
    print(f"  ✅ Boylam aralığı    : "
          f"{api_duraklar['BOYLAM'].min():.4f} → "
          f"{api_duraklar['BOYLAM'].max():.4f}")
    print(f"\n  Örnek duraklar:")
    print(api_duraklar[['DURAK_KODU','DURAK_ADI',
                         'ENLEM','BOYLAM']].head(5).to_string(index=False))

    # CSV olarak kaydet
    api_duraklar.to_csv(
        API_PATH + 'api_duraklar_koordinatli.csv', index=False)
    print(f"\n  ✅ CSV kaydedildi → api_duraklar_koordinatli.csv")
else:
    print("  ⚠️  Durak GeoJSON yok — D_DURAK tablosu kullanılacak")
    api_duraklar = None


# ── 5. GTFS stop_times Önizleme ───────────────────────────
print("\n📍 5. GTFS İçerik Kontrolü...")

gtfs_dosyalar = {
    'stop_times.txt': 'Durak Zamanları',
    'routes.txt':     'Hatlar',
    'stops.txt':      'Duraklar',
    'trips.txt':      'Seferler',
    'shapes.txt':     'Güzergah Şekilleri',
}

for dosya, aciklama in gtfs_dosyalar.items():
    yol = GTFS_PATH + dosya
    if os.path.exists(yol):
        mb = os.path.getsize(yol) / 1e6
        df_g = pd.read_csv(yol, nrows=2)
        print(f"  ✅ {aciklama:<25} {mb:>8.1f} MB  "
              f"kolonlar: {list(df_g.columns)[:5]}")
    else:
        print(f"  ❌ {aciklama:<25} bulunamadı")


# ── 6. Özet ──────────────────────────────────────────────
print("\n" + "=" * 60)
print("📊 API VERİ ÖZET")
print("=" * 60)
print(f"""
  Veri                      Durum      Modülde Kullanım
  ──────────────────────────────────────────────────────
  Otobüs Durakları GeoJSON  {'✅' if os.path.exists(durak_yolu) else '❌'}         Modül 3 & 5
  Hat Güzergahları GeoJSON  {'✅' if os.path.exists(hat_yolu) else '❌ (404)'}    Modül 3
  GTFS stop_times           {'✅' if os.path.exists(GTFS_PATH+'stop_times.txt') else '❌'}         Modül 3
  api_duraklar DataFrame    {'✅' if api_duraklar is not None else '❌'}         Modül 3 & 5
""")

gc.collect()
print("🎯 IBB API tamamlandı! → Modül 4 (Kriz Zaman Serisi)")

🌐 IBB OPEN DATA API — VERİ ÇEKME

📍 1. İETT Otobüs Durakları...
  ✅ Zaten mevcut: 7.1 MB

📍 2. İETT Hat Güzergahları...
  ✅ Zaten mevcut: 68.2 MB

📍 3. GTFS Verisi...
  Mevcut kaynaklar:
  CSV      | agency                         | https://data.ibb.gov.tr/dataset/8540e256-6df5-4719-85bc
  CSV      | calendar                       | https://data.ibb.gov.tr/dataset/8540e256-6df5-4719-85bc
  CSV      | routes                         | https://data.ibb.gov.tr/dataset/8540e256-6df5-4719-85bc
  CSV      | stop_times                     | https://data.ibb.gov.tr/dataset/8540e256-6df5-4719-85bc
  CSV      | stops                          | https://data.ibb.gov.tr/dataset/8540e256-6df5-4719-85bc
  CSV      | trips                          | https://data.ibb.gov.tr/dataset/8540e256-6df5-4719-85bc
  ZIP      | stop_times                     | https://data.ibb.gov.tr/dataset/8540e256-6df5-4719-85bc
  ⬇️  GTFS ZIP (ZIP) indiriliyor...
  ✅ GTFS ZIP: 1 dosya açıldı → ['stop_times.txt']

📍 4. Durak G

In [25]:
# ============================================================
# İETT SENTINEL — Hücre 7 MODÜL 4: KRİZ & ZAMAN SERİSİ ANALİZİ
# "Hava koşulları, saat ve mevsim sistemi nasıl etkiliyor?"
# ============================================================

import gc
import warnings
warnings.filterwarnings('ignore')
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

print("=" * 60)
print("🌪️  MODÜL 4: KRİZ & ZAMAN SERİSİ ANALİZİ")
print("=" * 60)

# ── ADIM 1: Kaza Zaman Serisi ─────────────────────────────
print("\n📅 ADIM 1: Kaza zaman serisi...")

aylik_kaza = (f_kaza_temiz
              .groupby(['KAZA_YIL','KAZA_AY'])
              .agg(kaza_sayisi     =('FID',              'count'),
                   toplam_yarali   =('TOPLAM_YARALI',    'sum'),
                   toplam_olum     =('TOPLAM_OLUM',      'sum'),
                   ciddi_kaza      =('CIDDI_KAZA',       'sum'),
                   kriz_tetikleyici=('KRİZ_TETIKLEYICI', 'sum'))
              .reset_index())

aylik_kaza['TARIH'] = pd.to_datetime(
    aylik_kaza['KAZA_YIL'].astype(str) + '-' +
    aylik_kaza['KAZA_AY'].astype(str) + '-01'
)
aylik_kaza = aylik_kaza.sort_values('TARIH')

print(f"  ✅ Toplam kaza       : {aylik_kaza['kaza_sayisi'].sum():,}")
print(f"  ✅ Toplam yaralı     : {aylik_kaza['toplam_yarali'].sum():,}")
print(f"  ✅ Kriz tetikleyici  : {aylik_kaza['kriz_tetikleyici'].sum():,}")

# ── ADIM 2: Arıza Zaman Serisi ────────────────────────────
print("\n📅 ADIM 2: Arıza zaman serisi...")

aylik_ariza = (f_ariza_temiz
               .groupby(['YIL','AY'])
               .agg(ariza_sayisi =('FID',            'count'),
                    zayi_sefer   =('ZAYISEFERSAYISI', 'sum'),
                    cekici_talep =('CEKICITALEP',      'sum'))
               .reset_index())

aylik_ariza['TARIH'] = pd.to_datetime(
    aylik_ariza['YIL'].astype(str) + '-' +
    aylik_ariza['AY'].astype(str) + '-01'
)
aylik_ariza = aylik_ariza.sort_values('TARIH')

print(f"  ✅ 2024 arıza : {aylik_ariza[aylik_ariza['YIL']==2024]['ariza_sayisi'].sum():,}")
print(f"  ✅ 2025 arıza : {aylik_ariza[aylik_ariza['YIL']==2025]['ariza_sayisi'].sum():,}")

# ── ADIM 3: Hava & Yol Durumu × Kaza ─────────────────────
print("\n🌤️  ADIM 3: Hava & yol durumu analizi...")

hava_kaza = (f_kaza_temiz
             .groupby('HAVA_DURUMU')
             .agg(kaza_sayisi =('FID',           'count'),
                  ort_hiz     =('HIZSON',         'mean'),
                  ciddi_kaza  =('CIDDI_KAZA',     'sum'),
                  yarali      =('TOPLAM_YARALI',  'sum'))
             .round(2)
             .sort_values('kaza_sayisi', ascending=False)
             .reset_index())

yol_kaza = (f_kaza_temiz
            .groupby('YOL_DURUMU')
            .agg(kaza_sayisi =('FID',         'count'),
                 ciddi_kaza  =('CIDDI_KAZA',  'sum'),
                 ort_hiz     =('HIZSON',       'mean'))
            .round(2)
            .sort_values('kaza_sayisi', ascending=False)
            .reset_index())

print(f"\n  Hava Durumu × Kaza:")
print(hava_kaza.to_string(index=False))
print(f"\n  Yol Durumu × Kaza:")
print(yol_kaza.to_string(index=False))

# ── ADIM 4: Saatlik Kriz Profili ─────────────────────────
print("\n🕐 ADIM 4: Saatlik kriz profili...")

saatlik_kaza = (f_kaza_temiz
                .groupby('KAZA_SAATI')
                .agg(kaza_sayisi =('FID',         'count'),
                     ciddi_kaza  =('CIDDI_KAZA',  'sum'),
                     ort_hiz     =('HIZSON',       'mean'))
                .reset_index()
                .sort_values('KAZA_SAATI'))

saatlik_ariza = (f_ariza_temiz
                 .groupby('SAAT')
                 .agg(ariza_sayisi =('FID',            'count'),
                      zayi_sefer   =('ZAYISEFERSAYISI', 'sum'))
                 .reset_index()
                 .sort_values('SAAT'))

pik_kaza  = saatlik_kaza.loc[
    saatlik_kaza['kaza_sayisi'].idxmax(), 'KAZA_SAATI']
pik_ariza = saatlik_ariza.loc[
    saatlik_ariza['ariza_sayisi'].idxmax(), 'SAAT']

print(f"  ✅ Kaza pik saati  : {pik_kaza:02d}:00")
print(f"  ✅ Arıza pik saati : {pik_ariza:02d}:00")

# ── ADIM 5: Kriz Günü Profili ─────────────────────────────
print("\n🚨 ADIM 5: Kriz günü profili...")

gunluk_kaza = (f_kaza_temiz
               .groupby('KAZA_GUNU')
               .agg(kaza_sayisi =('FID',          'count'),
                    ciddi_kaza  =('CIDDI_KAZA',   'sum'),
                    yarali      =('TOPLAM_YARALI', 'sum'))
               .reset_index())

gun_tr = {
    'Monday':'Pazartesi', 'Tuesday':'Salı',
    'Wednesday':'Çarşamba', 'Thursday':'Perşembe',
    'Friday':'Cuma', 'Saturday':'Cumartesi', 'Sunday':'Pazar'
}
gunluk_kaza['gun_tr'] = gunluk_kaza['KAZA_GUNU'].map(gun_tr)

print(f"\n  Güne Göre Kaza:")
print(gunluk_kaza[['gun_tr','kaza_sayisi','ciddi_kaza','yarali']]
      .to_string(index=False))

haftasonu_oran = f_kaza_temiz['HAFTASONU'].mean() * 100
print(f"\n  ✅ Haftasonu kaza oranı: %{haftasonu_oran:.1f}")

# ── ADIM 6: İlçe Bazlı Risk ───────────────────────────────
print("\n🗺️  ADIM 6: İlçe bazlı risk...")

ilce_kaza = (f_kaza_temiz
             .groupby('ILCE')
             .agg(kaza_sayisi   =('FID',              'count'),
                  ciddi_kaza    =('CIDDI_KAZA',        'sum'),
                  toplam_yarali =('TOPLAM_YARALI',     'sum'),
                  kriz_orani    =('KRİZ_TETIKLEYICI',  'mean'))
             .round(3)
             .sort_values('kaza_sayisi', ascending=False)
             .reset_index())

print(f"\n  En Riskli 15 İlçe:")
print(ilce_kaza[ilce_kaza['ILCE'] != 'BELİRSİZ']
      .head(15).to_string(index=False))

# ── ADIM 7: 2024 vs 2025 Karşılaştırma ───────────────────
print("\n📊 ADIM 7: 2024 vs 2025 karşılaştırması...")

k24 = f_kaza_temiz[f_kaza_temiz['KAZA_YIL'] == 2024]
k25 = f_kaza_temiz[f_kaza_temiz['KAZA_YIL'] == 2025]
a24 = f_ariza_temiz[f_ariza_temiz['YIL'] == 2024]
a25 = f_ariza_temiz[f_ariza_temiz['YIL'] == 2025]

print(f"\n  {'Metrik':<35} {'2024':>10} {'2025':>10} {'Değişim':>10}")
print(f"  {'─'*65}")

def karsilastir(isim, v24, v25):
    degisim = ((v25 - v24) / v24 * 100) if v24 > 0 else 0
    yon = '📈' if degisim > 0 else '📉'
    print(f"  {isim:<35} {v24:>10,.0f} {v25:>10,.0f} "
          f"{yon} %{abs(degisim):>5.1f}")

karsilastir('Toplam Kaza',        len(k24),                      len(k25))
karsilastir('Ciddi Kaza',         k24['CIDDI_KAZA'].sum(),       k25['CIDDI_KAZA'].sum())
karsilastir('Toplam Arıza',       len(a24),                      len(a25))
karsilastir('Zayi Sefer',         a24['ZAYISEFERSAYISI'].sum(),  a25['ZAYISEFERSAYISI'].sum())
karsilastir('Kriz Tetikleyici',   k24['KRİZ_TETIKLEYICI'].sum(), k25['KRİZ_TETIKLEYICI'].sum())

# ── ADIM 8: Görselleştirmeler ─────────────────────────────
print("\n📊 ADIM 8: Görselleştirmeler...")

# Grafik 1: Arıza & Kaza aylık trend
fig1 = make_subplots(
    rows=2, cols=1,
    subplot_titles=['Aylık Arıza Sayısı (2024 vs 2025)',
                    'Aylık Kaza Sayısı (2024 vs 2025)'],
    shared_xaxes=True
)
for yil, renk in [(2024,'#3498db'),(2025,'#e74c3c')]:
    d = aylik_ariza[aylik_ariza['YIL'] == yil]
    fig1.add_trace(go.Scatter(
        x=d['TARIH'], y=d['ariza_sayisi'],
        name=f'Arıza {yil}', line=dict(color=renk),
        mode='lines+markers'), row=1, col=1)

for yil, renk in [(2024,'#2ecc71'),(2025,'#f39c12')]:
    d = aylik_kaza[aylik_kaza['KAZA_YIL'] == yil]
    fig1.add_trace(go.Scatter(
        x=d['TARIH'], y=d['kaza_sayisi'],
        name=f'Kaza {yil}', line=dict(color=renk),
        mode='lines+markers'), row=2, col=1)

fig1.update_layout(template='plotly_dark', height=600,
                   title='📈 Arıza & Kaza Aylık Trend (2024 vs 2025)')
fig1.show()

# Grafik 2: Saatlik dağılım
fig2 = make_subplots(rows=1, cols=2,
                     subplot_titles=['Saate Göre Kaza',
                                     'Saate Göre Arıza'])
fig2.add_trace(go.Bar(x=saatlik_kaza['KAZA_SAATI'],
                      y=saatlik_kaza['kaza_sayisi'],
                      marker_color='#e74c3c', name='Kaza'),
               row=1, col=1)
fig2.add_trace(go.Bar(x=saatlik_ariza['SAAT'],
                      y=saatlik_ariza['ariza_sayisi'],
                      marker_color='#e67e22', name='Arıza'),
               row=1, col=2)
fig2.update_layout(template='plotly_dark', height=450,
                   title='🕐 Saatlik Kriz Profili', showlegend=False)
fig2.show()

# Grafik 3: Hava durumu × kaza
fig3 = px.bar(
    hava_kaza[hava_kaza['HAVA_DURUMU'] != 'BELİRSİZ'],
    x='HAVA_DURUMU', y='kaza_sayisi',
    color='ciddi_kaza', color_continuous_scale='Reds',
    title='🌤️ Hava Durumuna Göre Kaza Dağılımı',
    template='plotly_dark', width=800, height=450,
    labels={'HAVA_DURUMU':'Hava Durumu',
            'kaza_sayisi':'Kaza Sayısı',
            'ciddi_kaza':'Ciddi Kaza'}
)
fig3.show()

# Grafik 4: İlçe risk haritası
fig4 = px.bar(
    ilce_kaza[ilce_kaza['ILCE'] != 'BELİRSİZ'].head(15),
    x='ILCE', y='kaza_sayisi',
    color='kriz_orani', color_continuous_scale='RdYlGn_r',
    title='🗺️ İlçe Bazlı Kaza Risk Haritası (Top 15)',
    template='plotly_dark', width=900, height=480,
    labels={'ILCE':'İlçe', 'kaza_sayisi':'Kaza Sayısı',
            'kriz_orani':'Kriz Oranı'}
)
fig4.update_xaxes(tickangle=45)
fig4.show()

# ── Kaydet ───────────────────────────────────────────────
SAVE = '/content/drive/MyDrive/iett_sentinel/outputs/reports/'
ilce_kaza.to_csv(SAVE + 'modul4_ilce_risk.csv', index=False)
aylik_ariza.to_csv(SAVE + 'modul4_aylik_ariza_trend.csv', index=False)
aylik_kaza.to_csv(SAVE + 'modul4_aylik_kaza_trend.csv', index=False)

gc.collect()

print(f"\n✅ Sonuçlar kaydedildi → {SAVE}")
print(f"\n📊 MODÜL 4 ÖZET:")
print(f"   Kaza pik saati         : {pik_kaza:02d}:00")
print(f"   Arıza pik saati        : {pik_ariza:02d}:00")
print(f"   Haftasonu kaza oranı   : %{haftasonu_oran:.1f}")
en_riskli = ilce_kaza[ilce_kaza['ILCE']!='BELİRSİZ'].iloc[0]['ILCE']
print(f"   En riskli ilçe         : {en_riskli}")
print(f"   Kriz tetikleyici kaza  : {f_kaza_temiz['KRİZ_TETIKLEYICI'].sum():,}")
print(f"\n🎯 Modül 4 tamamlandı! → Modül 2B (Denetim-Arıza Korelasyonu)")

🌪️  MODÜL 4: KRİZ & ZAMAN SERİSİ ANALİZİ

📅 ADIM 1: Kaza zaman serisi...
  ✅ Toplam kaza       : 11,941
  ✅ Toplam yaralı     : 154
  ✅ Kriz tetikleyici  : 403

📅 ADIM 2: Arıza zaman serisi...
  ✅ 2024 arıza : 162,020
  ✅ 2025 arıza : 171,366

🌤️  ADIM 3: Hava & yol durumu analizi...

  Hava Durumu × Kaza:
HAVA_DURUMU  kaza_sayisi  ort_hiz  ciddi_kaza  yarali
    Güneşli         4256    10.88           3      60
    Bulutlu         2838     9.25           1      23
      Akşam         2410    10.78           3      42
   Yağmurlu          883    12.72           0      14
   BELİRSİZ          861    10.01           1       5
      Sabah          608    14.73           0       9
      Karlı           64     6.41           0       1
      Sisli           15    14.64           0       0
    Fırtına            6     2.00           0       0

  Yol Durumu × Kaza:
  YOL_DURUMU  kaza_sayisi  ciddi_kaza  ort_hiz
      Normal         9903           7    10.67
Yağış Kaygan         1117           


✅ Sonuçlar kaydedildi → /content/drive/MyDrive/iett_sentinel/outputs/reports/

📊 MODÜL 4 ÖZET:
   Kaza pik saati         : 16:00
   Arıza pik saati        : 16:00
   Haftasonu kaza oranı   : %22.5
   En riskli ilçe         : Sancaktepe
   Kriz tetikleyici kaza  : 403

🎯 Modül 4 tamamlandı! → Modül 2B (Denetim-Arıza Korelasyonu)


In [11]:
# ============================================================
# İETT SENTINEL — HÜCRE 8   MODÜL 2B: DENETİM-ARIZA KORELASYONU
# KAPINO → SID köprüsü ile doğru join
# ============================================================

import gc
import warnings
warnings.filterwarnings('ignore')
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

print("=" * 60)
print("🔍 MODÜL 2B: DENETİM-ARIZA KORELASYONU")
print("=" * 60)

# ── KÖPRÜ: KAPINO → SID ───────────────────────────────────
print("\n🌉 KÖPRÜ: d_arac KAPINO → SID eşlemesi...")

kapino_sid = d_arac[['SID','KAPINO']].dropna().copy()
kapino_sid['KAPINO'] = kapino_sid['KAPINO'].astype(str).str.strip().str.upper()
kapino_sid['SID']    = kapino_sid['SID'].astype(int)
print(f"  ✅ Köprü boyutu: {len(kapino_sid):,} araç")
print(f"  ✅ Örnek: {kapino_sid.head(3).values.tolist()}")

# ── ADIM 1: Denetim verisini hazırla ─────────────────────
print("\n📦 ADIM 1: Denetim verisi hazırlanıyor...")

f_den_arac = f_denetim[
    f_denetim['VARLIKTIP'].astype(str).str.upper().str.contains('ARA', na=False)
].copy()

_tar_raw = f_den_arac['SIDKAYITTARIHI']
# Unix timestamp (integer) mi, normal tarih string mi?
if pd.to_numeric(_tar_raw, errors='coerce').notna().mean() > 0.8:
    # Sayısal → Unix ms olarak dene, tutarsız gelirse unit='s' dene
    _parsed = pd.to_datetime(pd.to_numeric(_tar_raw, errors='coerce'),
                             unit='ms', errors='coerce')
    # 2020 öncesi tarih çok fazlaysa unit='s' dene
    if (_parsed.dt.year < 2020).mean() > 0.5:
        _parsed = pd.to_datetime(pd.to_numeric(_tar_raw, errors='coerce'),
                                 unit='s', errors='coerce')
    # Hâlâ 2020 öncesiyse doğrudan string parse dene
    if (_parsed.dt.year < 2020).mean() > 0.5:
        _parsed = pd.to_datetime(_tar_raw.astype(str), errors='coerce')
    f_den_arac['SIDKAYITTARIHI'] = _parsed
else:
    f_den_arac['SIDKAYITTARIHI'] = pd.to_datetime(_tar_raw, errors='coerce')

print(f"  ✅ Tarih aralığı: {f_den_arac['SIDKAYITTARIHI'].min()} → "
      f"{f_den_arac['SIDKAYITTARIHI'].max()}")
f_den_arac['VARLIKKODU_UP'] = (f_den_arac['VARLIKKODU']
                                .astype(str).str.strip().str.upper())

# KAPINO → SID join
f_den_arac = f_den_arac.merge(
    kapino_sid.rename(columns={'KAPINO':'VARLIKKODU_UP','SID':'ARAC_SID'}),
    on='VARLIKKODU_UP', how='left'
)

eslesen = f_den_arac['ARAC_SID'].notna().sum()
print(f"  ✅ Araç denetimi      : {len(f_den_arac):,} kayıt")
print(f"  ✅ SID eşleşen kayıt  : {eslesen:,} ({eslesen/len(f_den_arac)*100:.1f}%)")

# Araç bazında denetim özeti
denetim_ozet = (f_den_arac[f_den_arac['ARAC_SID'].notna()]
    .groupby('ARAC_SID')
    .agg(
        toplam_denetim  =('FID',            'count'),
        olumsuz_denetim =('DENETIMDURUM',   lambda x: (x=='Olumsuz').sum()),
        olumlu_denetim  =('DENETIMDURUM',   lambda x: (x=='Olumlu').sum()),
        ilk_denetim_tar =('SIDKAYITTARIHI', 'min'),
        son_denetim_tar =('SIDKAYITTARIHI', 'max'),
    )
    .reset_index()
)
denetim_ozet['OLUMSUZ_ORAN'] = (
    denetim_ozet['olumsuz_denetim'] / denetim_ozet['toplam_denetim']
).round(3)

print(f"  ✅ Benzersiz araç     : {len(denetim_ozet):,}")
print(f"  ✅ Olumsuz oran       : "
      f"Ort %{denetim_ozet['OLUMSUZ_ORAN'].mean()*100:.1f}  |  "
      f"Max %{denetim_ozet['OLUMSUZ_ORAN'].max()*100:.1f}")

# ── ADIM 2: Son Olumsuz Denetim Tarihi ───────────────────
print("\n📅 ADIM 2: Son olumsuz denetim tarihi...")

son_olumsuz = (f_den_arac[
        (f_den_arac['DENETIMDURUM'] == 'Olumsuz') &
        (f_den_arac['ARAC_SID'].notna())
    ]
    .groupby('ARAC_SID')['SIDKAYITTARIHI']
    .max()
    .reset_index()
    .rename(columns={'SIDKAYITTARIHI':'SON_OLUMSUZ_DENETIM',
                     'ARAC_SID':'SIDARAC'})
)
son_olumsuz['SIDARAC'] = son_olumsuz['SIDARAC'].astype(float)
print(f"  ✅ Olumsuz denetim alan araç: {len(son_olumsuz):,}")

# ── ADIM 3: Denetim → Arıza Gecikme ──────────────────────
print("\n⏱️  ADIM 3: Denetim → Arıza gecikme analizi...")

ariza_sonrasi = f_ariza_temiz.merge(son_olumsuz, on='SIDARAC', how='inner')

ariza_sonrasi['GECIKME_GUN'] = (
    ariza_sonrasi['OLAYTARIHI'] - ariza_sonrasi['SON_OLUMSUZ_DENETIM']
).dt.days

ariza_pozitif = ariza_sonrasi[
    ariza_sonrasi['GECIKME_GUN'].between(0, 180)
].copy()

print(f"  ✅ Eşleşen arıza kaydı      : {len(ariza_pozitif):,}")
if len(ariza_pozitif) > 0:
    print(f"  ✅ Ortalama gecikme         : {ariza_pozitif['GECIKME_GUN'].mean():.1f} gün")
    print(f"  ✅ Medyan gecikme           : {ariza_pozitif['GECIKME_GUN'].median():.1f} gün")
    print(f"  ✅ 14 gün içinde arıza      : %{(ariza_pozitif['GECIKME_GUN']<=14).mean()*100:.1f}")
    print(f"  ✅ 30 gün içinde arıza      : %{(ariza_pozitif['GECIKME_GUN']<=30).mean()*100:.1f}")
else:
    print("  ⚠️  Hâlâ eşleşme yok — tarih aralığı kontrolü:")
    print(f"     f_ariza OLAYTARIHI : {f_ariza_temiz['OLAYTARIHI'].min()} → {f_ariza_temiz['OLAYTARIHI'].max()}")
    print(f"     son_olumsuz tarih  : {son_olumsuz['SON_OLUMSUZ_DENETIM'].min()} → {son_olumsuz['SON_OLUMSUZ_DENETIM'].max()}")

# ── ADIM 4: Arıza Türü × Gecikme ─────────────────────────
print("\n🔧 ADIM 4: Arıza türü × gecikme...")

if len(ariza_pozitif) > 0:
    tur_gecikme = (ariza_pozitif
        .groupby('ARIZAUSTKODTANIM')
        .agg(
            ariza_sayisi  =('FID',         'count'),
            ort_gecikme   =('GECIKME_GUN', 'mean'),
            medyan_gecikme=('GECIKME_GUN', 'median'),
            erken_ariza   =('GECIKME_GUN', lambda x: (x<=14).sum()),
        )
        .round(1)
        .sort_values('ariza_sayisi', ascending=False)
        .head(15)
        .reset_index()
    )
    tur_gecikme['ERKEN_ORAN'] = (
        tur_gecikme['erken_ariza'] / tur_gecikme['ariza_sayisi'] * 100
    ).round(1)
    print(tur_gecikme[['ARIZAUSTKODTANIM','ariza_sayisi',
                        'ort_gecikme','medyan_gecikme','ERKEN_ORAN']]
          .to_string(index=False))
else:
    tur_gecikme = pd.DataFrame()
    print("  ⚠️  Veri yok, atlanıyor")

# ── ADIM 5: Korelasyon ───────────────────────────────────
print("\n📊 ADIM 5: Denetim × Arıza korelasyonu...")

arac_ariza_sayisi = (f_ariza_temiz
    .groupby('SIDARAC')
    .agg(toplam_ariza=('FID','count'),
         zayi_sefer  =('ZAYISEFERSAYISI','sum'))
    .reset_index()
)
denetim_ozet_join = denetim_ozet.rename(columns={'ARAC_SID':'SIDARAC'})
denetim_ozet_join['SIDARAC'] = denetim_ozet_join['SIDARAC'].astype(float)

denetim_ariza = denetim_ozet_join.merge(
    arac_ariza_sayisi, on='SIDARAC', how='inner'
)
print(f"  ✅ Korelasyon için eşleşen araç: {len(denetim_ariza):,}")

if len(denetim_ariza) > 1:
    korel  = denetim_ariza['OLUMSUZ_ORAN'].corr(denetim_ariza['toplam_ariza'])
    korel2 = denetim_ariza['olumsuz_denetim'].corr(denetim_ariza['toplam_ariza'])
    print(f"  Olumsuz Oran   × Arıza Sayısı : r = {korel:.3f}")
    print(f"  Olumsuz Sayısı × Arıza Sayısı : r = {korel2:.3f}")
else:
    korel, korel2 = 0, 0
    print("  ⚠️  Yeterli eşleşme yok")

# ── ADIM 6: Denetim Tipi Analizi ─────────────────────────
print("\n🏷️  ADIM 6: Denetim tipi analizi...")

tip_ozet = (f_den_arac
    .groupby('SIDDENETIMTIP')
    .agg(toplam =('FID','count'),
         olumsuz=('DENETIMDURUM', lambda x: (x=='Olumsuz').sum()))
    .reset_index()
)
tip_ozet['olumsuz_oran'] = (
    tip_ozet['olumsuz'] / tip_ozet['toplam'] * 100
).round(1)
tip_ozet = tip_ozet.sort_values('olumsuz_oran', ascending=False).head(10)

# D_DENETIMTIP ile join
if d_denetimtip is not None:
    tip_ozet = tip_ozet.merge(
        d_denetimtip[['SID','DENETIMADI']],
        left_on='SIDDENETIMTIP', right_on='SID', how='left'
    )
    print(tip_ozet[['DENETIMADI','toplam','olumsuz','olumsuz_oran']]
          .to_string(index=False))
else:
    print(tip_ozet.to_string(index=False))

# ── ADIM 7: Görselleştirmeler ─────────────────────────────
print("\n📊 ADIM 7: Görselleştirmeler...")

# Grafik 1: Olumsuz denetim oranı dağılımı (her zaman çizilir)
fig4 = px.histogram(
    denetim_ozet[denetim_ozet['toplam_denetim'] >= 5],
    x='OLUMSUZ_ORAN', nbins=40,
    title='📊 Araçların Olumsuz Denetim Oranı Dağılımı',
    template='plotly_dark', color_discrete_sequence=['#e67e22'],
    labels={'OLUMSUZ_ORAN':'Olumsuz Denetim Oranı'},
    width=800, height=420
)
fig4.show()

# Grafik 2: Denetim tipi × olumsuz oran
fig5 = px.bar(
    tip_ozet.sort_values('olumsuz_oran', ascending=True),
    x='olumsuz_oran',
    y=tip_ozet.get('DENETIMADI', tip_ozet['SIDDENETIMTIP']).astype(str),
    orientation='h',
    title='🏷️ Denetim Tipine Göre Olumsuz Oran (%)',
    template='plotly_dark', color='olumsuz_oran',
    color_continuous_scale='RdYlGn_r',
    width=850, height=450,
    labels={'x':'Olumsuz Oran (%)','y':'Denetim Tipi'}
)
fig5.show()

if len(ariza_pozitif) > 0:
    # Grafik 3: Gecikme histogramı
    fig1 = px.histogram(
        ariza_pozitif[ariza_pozitif['GECIKME_GUN'] <= 90],
        x='GECIKME_GUN', nbins=45,
        title='⏱️ Olumsuz Denetimden Sonra Kaç Gün İçinde Arıza? (0-90 gün)',
        template='plotly_dark', color_discrete_sequence=['#e74c3c'],
        labels={'GECIKME_GUN':'Gecikme (Gün)'},
        width=900, height=450
    )
    fig1.add_vline(x=14, line_dash='dash', line_color='yellow',
                   annotation_text='14 gün')
    fig1.add_vline(x=30, line_dash='dash', line_color='orange',
                   annotation_text='30 gün')
    fig1.show()

    # Grafik 4: Korelasyon scatter
    if len(denetim_ariza) > 1:
        fig3 = px.scatter(
            denetim_ariza.sample(min(2000, len(denetim_ariza))),
            x='OLUMSUZ_ORAN', y='toplam_ariza',
            color='zayi_sefer', color_continuous_scale='Reds',
            title=f'📊 Olumsuz Denetim Oranı × Arıza Sayısı (r={korel:.3f})',
            template='plotly_dark', width=850, height=500,
            labels={'OLUMSUZ_ORAN':'Olumsuz Denetim Oranı',
                    'toplam_ariza':'Toplam Arıza',
                    'zayi_sefer':'Zayi Sefer'}
        )
        fig3.show()

    if len(tur_gecikme) > 0:
        # Grafik 5: Arıza türü × gecikme
        fig2 = px.bar(
            tur_gecikme.sort_values('ort_gecikme'),
            x='ort_gecikme', y='ARIZAUSTKODTANIM', orientation='h',
            color='ERKEN_ORAN', color_continuous_scale='RdYlGn_r',
            title='🔧 Arıza Türüne Göre Denetim→Arıza Gecikme (Gün)',
            template='plotly_dark', width=900, height=500,
            labels={'ARIZAUSTKODTANIM':'Arıza Türü',
                    'ort_gecikme':'Ort. Gecikme (Gün)',
                    'ERKEN_ORAN':'14 Gün İçi %'}
        )
        fig2.show()

# ── Kaydet ───────────────────────────────────────────────
SAVE = '/content/drive/MyDrive/iett_sentinel/outputs/reports/'
denetim_ozet.to_csv(SAVE + 'modul2b_denetim_ozet.csv', index=False)
if len(ariza_pozitif) > 0:
    ariza_pozitif[['SIDARAC','OLAYTARIHI','ARIZAUSTKODTANIM',
                   'GECIKME_GUN','SON_OLUMSUZ_DENETIM']]\
        .to_csv(SAVE + 'modul2b_gecikme.csv', index=False)

gc.collect()
print(f"\n✅ Sonuçlar kaydedildi → {SAVE}")
print(f"\n📊 MODÜL 2B ÖZET:")
print(f"   Olumsuz denetim alan araç  : {len(son_olumsuz):,}")
print(f"   Korelasyon için eşleşen    : {len(denetim_ariza):,}")
if len(ariza_pozitif) > 0:
    print(f"   Denetim sonrası arıza      : {len(ariza_pozitif):,}")
    print(f"   Ortalama gecikme           : {ariza_pozitif['GECIKME_GUN'].mean():.1f} gün")
    print(f"   14 gün içi arıza oranı     : %{(ariza_pozitif['GECIKME_GUN']<=14).mean()*100:.1f}")
print(f"   Denetim × Arıza korelasyon : r={korel:.3f}")
print(f"\n🎯 Modül 2B tamamlandı! → Modül 3 (Domino Etkisi)")

🔍 MODÜL 2B: DENETİM-ARIZA KORELASYONU

🌉 KÖPRÜ: d_arac KAPINO → SID eşlemesi...
  ✅ Köprü boyutu: 8,684 araç
  ✅ Örnek: [[1, 'O1001'], [2, 'O1002'], [3, 'O1003']]

📦 ADIM 1: Denetim verisi hazırlanıyor...
  ✅ Tarih aralığı: 2024-01-01 00:00:00 → 2025-12-31 00:00:00
  ✅ Araç denetimi      : 8,379,578 kayıt
  ✅ SID eşleşen kayıt  : 8,379,578 (100.0%)
  ✅ Benzersiz araç     : 6,787
  ✅ Olumsuz oran       : Ort %21.9  |  Max %100.0

📅 ADIM 2: Son olumsuz denetim tarihi...
  ✅ Olumsuz denetim alan araç: 6,650

⏱️  ADIM 3: Denetim → Arıza gecikme analizi...
  ✅ Eşleşen arıza kaydı      : 4,564
  ✅ Ortalama gecikme         : 15.0 gün
  ✅ Medyan gecikme           : 6.0 gün
  ✅ 14 gün içinde arıza      : %71.3
  ✅ 30 gün içinde arıza      : %86.5

🔧 ADIM 4: Arıza türü × gecikme...
                ARIZAUSTKODTANIM  ariza_sayisi  ort_gecikme  medyan_gecikme  ERKEN_ORAN
      ELEKTRİK SİSTEMİ ARIZALARI           401        17.80            9.00       65.60
                  KAPI ARIZALARI         


✅ Sonuçlar kaydedildi → /content/drive/MyDrive/iett_sentinel/outputs/reports/

📊 MODÜL 2B ÖZET:
   Olumsuz denetim alan araç  : 6,650
   Korelasyon için eşleşen    : 6,751
   Denetim sonrası arıza      : 4,564
   Ortalama gecikme           : 15.0 gün
   14 gün içi arıza oranı     : %71.3
   Denetim × Arıza korelasyon : r=0.545

🎯 Modül 2B tamamlandı! → Modül 3 (Domino Etkisi)


In [29]:
# ============================================================
# İETT SENTINEL — HÜCRE 9   MODÜL 3: KRİZ DOMINO ETKİSİ
# "Bir hat kapanırsa hangi hatlar etkilenir?"
# F_SEFER + F_YOLCULUK (SIDONCEKIHAT) + GTFS
# ============================================================

import gc
import warnings
warnings.filterwarnings('ignore')

!pip install -q networkx

import networkx as nx
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import json, os

print("=" * 60)
print("🕸️  MODÜL 3: KRİZ DOMINO ETKİSİ")
print("=" * 60)

GTFS_PATH = BASE_PATH + 'API_DATA/GTFS/'
SAVE      = '/content/drive/MyDrive/iett_sentinel/outputs/reports/'

# ── ADIM 1: Hat Bağımlılık Ağı — F_SEFER üzerinden ──────
print("\n🔗 ADIM 1: Hat bağımlılık ağı (F_SEFER)...")

# ── FIX: KILAVUZID string karşılaştırması ────────────────
if d_kilavuz is not None:
    # KILAVUZID string veya integer olabilir → her ikisini yakala
    mask_5 = d_kilavuz['KILAVUZID'].astype(str).str.strip() == '5'
    durum_df = d_kilavuz[mask_5][['SID','TANIM','TANIMDETAY']]
    print(f"  KILAVUZID=5 kayıt sayısı: {len(durum_df)}")
    print(f"  Durum değerleri: {durum_df['TANIM'].unique().tolist()}")
    durum_map = durum_df.set_index('SID')['TANIM'].to_dict()
else:
    durum_map = {}

# Sefer verisini zenginleştir
f_sefer_c = f_sefer.copy()
f_sefer_c['DURUM_ADI'] = f_sefer_c['SIDDURUMDEGISIKLIK'].map(durum_map).fillna('Normal')

# İptal tespiti — 'Normal' dışındaki tüm durum değişiklikleri iptal/aksaklık
f_sefer_c['IPTAL'] = (f_sefer_c['DURUM_ADI'] != 'Normal').astype(int)

print(f"  ✅ Toplam sefer       : {len(f_sefer_c):,}")
print(f"  ✅ İptal/arıza sefer  : {f_sefer_c['IPTAL'].sum():,} "
      f"(%{f_sefer_c['IPTAL'].mean()*100:.1f})")
print(f"  ✅ Durum türleri      : {f_sefer_c['DURUM_ADI'].value_counts().head(10).to_dict()}")

# ── ADIM 2: Hat Bazında İptal Profili ────────────────────
print("\n📊 ADIM 2: Hat bazında iptal profili...")

hat_iptal = (f_sefer_c
    .groupby('SIDHAT')
    .agg(
        toplam_sefer =('FID',    'count'),
        iptal_sefer  =('IPTAL',  'sum'),
    )
    .reset_index()
)
hat_iptal['IPTAL_ORANI'] = (
    hat_iptal['iptal_sefer'] / hat_iptal['toplam_sefer']
).round(3)

# D_HAT ile join
hat_iptal = hat_iptal.merge(
    d_hat_temiz[['SID','HATKODU','HATADI','HATCINSI','HATILCE']],
    left_on='SIDHAT', right_on='SID', how='left'
)

print(f"  ✅ Analiz edilen hat  : {len(hat_iptal):,}")
print(f"  ✅ En yüksek iptal oranı:")
print(hat_iptal.sort_values('IPTAL_ORANI', ascending=False)
      [['HATKODU','HATADI','HATILCE','toplam_sefer','iptal_sefer','IPTAL_ORANI']]
      .head(10).to_string(index=False))

# ── ADIM 3: GTFS stop_times → Hat-Durak Ağı ─────────────
print("\n🚌 ADIM 3: GTFS hat-durak ağı...")

gtfs_stop_times_yol = GTFS_PATH + 'stop_times.txt'
gtfs_trips_yol      = GTFS_PATH + 'trips.txt'
gtfs_routes_yol     = GTFS_PATH + 'routes.txt'

def gtfs_oku(yol, nrows=None):
    try:
        df = pd.read_csv(yol, nrows=nrows, low_memory=False)
        if df.shape[1] == 1:
            df = pd.read_csv(yol, sep=';', nrows=nrows, low_memory=False)
        df.columns = df.columns.str.strip().str.replace(';','')
        return df
    except Exception as e:
        print(f"  ❌ {yol}: {e}")
        return None

gtfs_trips  = gtfs_oku(gtfs_trips_yol)
gtfs_routes = gtfs_oku(gtfs_routes_yol)

if gtfs_trips is not None and gtfs_routes is not None:
    print(f"  ✅ trips  : {gtfs_trips.shape}")
    print(f"  ✅ routes : {gtfs_routes.shape}")

    trip_route = gtfs_trips[['trip_id','route_id']].drop_duplicates()

    print(f"\n  stop_times chunk okuma...")
    stop_times_chunks = []
    for chunk in pd.read_csv(gtfs_stop_times_yol, chunksize=500_000, low_memory=False):
        if chunk.shape[1] == 1:
            chunk = pd.read_csv(gtfs_stop_times_yol, sep=';',
                                nrows=500_000, low_memory=False)
        chunk.columns = chunk.columns.str.strip()
        stop_times_chunks.append(
            chunk[['trip_id','stop_id','stop_sequence']].head(500_000))
        if sum(len(c) for c in stop_times_chunks) > 2_000_000:
            break

    gtfs_st = pd.concat(stop_times_chunks, ignore_index=True)
    del stop_times_chunks; gc.collect()
    print(f"  ✅ stop_times : {len(gtfs_st):,} kayıt")

    durak_hat = (gtfs_st
        .merge(trip_route, on='trip_id', how='left')
        [['stop_id','route_id']]
        .drop_duplicates()
    )
    print(f"  ✅ Durak-hat ilişkisi: {len(durak_hat):,}")

    durak_hat_grp = durak_hat.groupby('stop_id')['route_id'].apply(list)
    hat_cift_sayac = {}
    for durak, hatlar in durak_hat_grp.items():
        hatlar = list(set(hatlar))
        if len(hatlar) < 2:
            continue
        for i in range(len(hatlar)):
            for j in range(i+1, len(hatlar)):
                key = tuple(sorted([str(hatlar[i]), str(hatlar[j])]))
                hat_cift_sayac[key] = hat_cift_sayac.get(key, 0) + 1

    print(f"  ✅ Hat çifti (ortak durak): {len(hat_cift_sayac):,}")
    gtfs_ok = True
else:
    print("  ⚠️  GTFS yüklenemedi — F_SEFER ile devam edilecek")
    hat_cift_sayac = {}
    gtfs_ok = False

# ── ADIM 4: NetworkX Graf Oluştur ────────────────────────
print("\n🕸️  ADIM 4: Hat bağımlılık grafı oluşturuluyor...")

G = nx.Graph()

if gtfs_ok and hat_cift_sayac:
    for (h1, h2), agirlik in hat_cift_sayac.items():
        if agirlik >= 3:
            G.add_edge(h1, h2, weight=agirlik)
    print(f"  ✅ GTFS bazlı graf: {G.number_of_nodes()} düğüm, "
          f"{G.number_of_edges()} kenar")
else:
    sefer_hat = (f_sefer_c[['SIDGUZERGAH','SIDHAT']]
                 .dropna().drop_duplicates())
    guz_hat = sefer_hat.groupby('SIDGUZERGAH')['SIDHAT'].apply(list)
    for guz, hatlar in guz_hat.items():
        hatlar = list(set([str(int(h)) for h in hatlar if pd.notna(h)]))
        if len(hatlar) < 2:
            continue
        for i in range(len(hatlar)):
            for j in range(i+1, len(hatlar)):
                h1, h2 = hatlar[i], hatlar[j]
                if G.has_edge(h1, h2):
                    G[h1][h2]['weight'] += 1
                else:
                    G.add_edge(h1, h2, weight=1)
    print(f"  ✅ Sefer bazlı graf: {G.number_of_nodes()} düğüm, "
          f"{G.number_of_edges()} kenar")

# ── ADIM 5: Merkezi Hat Analizi ───────────────────────────
print("\n🎯 ADIM 5: Merkezi hat analizi (en kritik hatlar)...")

if G.number_of_nodes() > 0:
    degree_cent = nx.degree_centrality(G)
    betweenness = nx.betweenness_centrality(G, k=min(50, G.number_of_nodes()))

    merkezi_df = pd.DataFrame({
        'hat_id'      : list(degree_cent.keys()),
        'degree_cent' : list(degree_cent.values()),
        'betweenness' : [betweenness.get(k, 0) for k in degree_cent.keys()],
        'derece'      : [G.degree(k) for k in degree_cent.keys()],
    }).sort_values('betweenness', ascending=False).head(20)

    if gtfs_ok and gtfs_routes is not None:
        route_isim = gtfs_routes.set_index(
            gtfs_routes.columns[0])[gtfs_routes.columns[2]].to_dict()
        merkezi_df['hat_adi'] = merkezi_df['hat_id'].map(
            lambda x: route_isim.get(x, x))
    else:
        hat_sid_adi = d_hat_temiz.set_index('SID')['HATKODU'].to_dict()
        merkezi_df['hat_adi'] = merkezi_df['hat_id'].map(
            lambda x: hat_sid_adi.get(int(x) if str(x).isdigit() else x, x))

    print(f"\n  🔴 En Kritik 20 Hat (Betweenness Centrality):")
    print(merkezi_df[['hat_adi','derece','degree_cent','betweenness']]
          .round(4).to_string(index=False))

# ── ADIM 6: Domino Simülasyonu ────────────────────────────
print("\n🚨 ADIM 6: Domino simülasyonu — Hat kapanırsa ne olur?")

def domino_simule(G, hat_id, derinlik=2):
    if hat_id not in G:
        return [], {}
    etkilenen = set()
    sinir = {hat_id}
    for d in range(derinlik):
        yeni = set()
        for h in sinir:
            komsular = set(G.neighbors(h)) - etkilenen - {hat_id}
            yeni.update(komsular)
        etkilenen.update(yeni)
        sinir = yeni
    agirliklar = {}
    for h in etkilenen:
        if G.has_edge(hat_id, h):
            agirliklar[h] = G[hat_id][h]['weight']
        else:
            agirliklar[h] = 0
    return list(etkilenen), agirliklar

if G.number_of_nodes() > 0:
    kritik_hatlar = merkezi_df.head(5)['hat_id'].tolist()
    print(f"\n  Simüle edilen kritik hatlar: {kritik_hatlar[:5]}")

    sim_sonuclar = []
    for hat in kritik_hatlar:
        etkilenen, agirliklar = domino_simule(G, hat, derinlik=2)
        hat_adi = merkezi_df[merkezi_df['hat_id']==hat]['hat_adi'].values
        hat_adi = hat_adi[0] if len(hat_adi) > 0 else str(hat)
        sim_sonuclar.append({
            'hat_id'         : hat,
            'hat_adi'        : hat_adi,
            'etkilenen_hat'  : len(etkilenen),
            'max_ortak_durak': max(agirliklar.values()) if agirliklar else 0,
        })
        print(f"\n  🔴 Hat {hat_adi} kapanırsa:")
        print(f"     → {len(etkilenen)} hat etkilenir")
        print(f"     → En güçlü bağlantı: "
              f"{max(agirliklar.values()) if agirliklar else 0} ortak durak")

    sim_df = pd.DataFrame(sim_sonuclar)

# ── ADIM 7: Görselleştirmeler ─────────────────────────────
print("\n📊 ADIM 7: Görselleştirmeler...")

# Grafik 1: Hat iptal oranı
fig1 = px.bar(
    hat_iptal[hat_iptal['toplam_sefer'] >= 50]
    .sort_values('IPTAL_ORANI', ascending=False).head(20),
    x='HATKODU', y='IPTAL_ORANI',
    color='iptal_sefer', color_continuous_scale='Reds',
    title='🚌 Hat Bazında İptal Oranı (Top 20)',
    template='plotly_dark', width=900, height=450,
    labels={'HATKODU':'Hat Kodu','IPTAL_ORANI':'İptal Oranı',
            'iptal_sefer':'İptal Sefer'}
)
fig1.update_xaxes(tickangle=45)
fig1.show()

if G.number_of_nodes() > 0:
    # Grafik 2: Betweenness centrality
    fig2 = px.bar(
        merkezi_df.sort_values('betweenness', ascending=True),
        x='betweenness', y='hat_adi', orientation='h',
        color='derece', color_continuous_scale='RdYlGn_r',
        title='🎯 En Kritik Hatlar — Betweenness Centrality',
        template='plotly_dark', width=900, height=550,
        labels={'betweenness':'Betweenness','hat_adi':'Hat',
                'derece':'Bağlantı Sayısı'}
    )
    fig2.show()

    # Grafik 3: Ağ grafiği
    top_nodes = [n for n, _ in sorted(degree_cent.items(),
                  key=lambda x: x[1], reverse=True)[:50]]
    subG = G.subgraph(top_nodes)
    pos  = nx.spring_layout(subG, seed=42, k=0.5)

    edge_x, edge_y = [], []
    for u, v in subG.edges():
        x0, y0 = pos[u]; x1, y1 = pos[v]
        edge_x += [x0, x1, None]; edge_y += [y0, y1, None]

    node_x   = [pos[n][0] for n in subG.nodes()]
    node_y   = [pos[n][1] for n in subG.nodes()]
    node_deg = [subG.degree(n) for n in subG.nodes()]
    node_lbl = [str(merkezi_df[merkezi_df['hat_id']==n]['hat_adi'].values[0])
                if n in merkezi_df['hat_id'].values else str(n)
                for n in subG.nodes()]

    fig3 = go.Figure()
    fig3.add_trace(go.Scatter(x=edge_x, y=edge_y, mode='lines',
        line=dict(width=0.5, color='#555'), hoverinfo='none'))
    fig3.add_trace(go.Scatter(x=node_x, y=node_y, mode='markers+text',
        marker=dict(size=[max(5, d*2) for d in node_deg],
                    color=node_deg, colorscale='RdYlGn_r',
                    showscale=True, colorbar=dict(title='Bağlantı')),
        text=node_lbl, textposition='top center',
        textfont=dict(size=7), hovertext=node_lbl, hoverinfo='text'))
    fig3.update_layout(
        title='🕸️ İETT Hat Bağımlılık Ağı (Top 50 Hat)',
        template='plotly_dark', width=950, height=700, showlegend=False,
        xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        yaxis=dict(showgrid=False, zeroline=False, showticklabels=False))
    fig3.show()

    # Grafik 4: Domino simülasyon
    if len(sim_df) > 0:
        fig4 = px.bar(
            sim_df.sort_values('etkilenen_hat', ascending=False),
            x='hat_adi', y='etkilenen_hat',
            color='max_ortak_durak', color_continuous_scale='Reds',
            title='🚨 Hat Kapanırsa Kaç Hat Etkilenir? (Domino)',
            template='plotly_dark', width=800, height=430,
            labels={'hat_adi':'Hat','etkilenen_hat':'Etkilenen Hat Sayısı',
                    'max_ortak_durak':'Max Ortak Durak'}
        )
        fig4.show()

# ── Kaydet ───────────────────────────────────────────────
hat_iptal.to_csv(SAVE + 'modul3_hat_iptal_profili.csv', index=False)
if G.number_of_nodes() > 0:
    merkezi_df.to_csv(SAVE + 'modul3_kritik_hatlar.csv', index=False)
    sim_df.to_csv(SAVE + 'modul3_domino_simulasyon.csv', index=False)

gc.collect()
print(f"\n✅ Sonuçlar kaydedildi → {SAVE}")
print(f"\n📊 MODÜL 3 ÖZET:")
print(f"   Analiz edilen hat     : {len(hat_iptal):,}")
print(f"   İptal oranı (ort)     : %{hat_iptal['IPTAL_ORANI'].mean()*100:.1f}")
if G.number_of_nodes() > 0:
    print(f"   Graf düğüm sayısı     : {G.number_of_nodes():,}")
    print(f"   Graf kenar sayısı     : {G.number_of_edges():,}")
    print(f"   En kritik hat         : {merkezi_df.iloc[0]['hat_adi']}")
    print(f"   Max domino etkisi     : {sim_df['etkilenen_hat'].max()} hat")
print(f"\n🎯 Modül 3 tamamlandı! → Modül 5 (Etkinlik Anomalisi)")

🕸️  MODÜL 3: KRİZ DOMINO ETKİSİ

🔗 ADIM 1: Hat bağımlılık ağı (F_SEFER)...
  KILAVUZID=5 kayıt sayısı: 50
  Durum değerleri: ['Personel', 'Araç', 'Trafik', 'Müdahale tipi']
  ✅ Toplam sefer       : 15,022,671
  ✅ İptal/arıza sefer  : 1,582,542 (%10.5)
  ✅ Durum türleri      : {'Normal': 13440129, 'Araç': 1241048, 'Trafik': 219235, 'Müdahale tipi': 62737, 'Personel': 59522}

📊 ADIM 2: Hat bazında iptal profili...
  ✅ Analiz edilen hat  : 870
  ✅ En yüksek iptal oranı:
HATKODU                                     HATADI     HATILCE  toplam_sefer  iptal_sefer  IPTAL_ORANI
     94           İBNİ SİNA İLKOKULU - ZEYTİNBURNU    Bakırköy             1            1         1.00
   145M                           BEYKENT - TAKSİM  Beylikdüzü             2            2         1.00
    19D                        ÜMRANİYE - BOSTANCI    Ümraniye            19           18         0.95
  130ŞT          ŞİFA MAHALLESİ - TAVŞANTEPE METRO       Tuzla            66           61         0.92
   500L      


✅ Sonuçlar kaydedildi → /content/drive/MyDrive/iett_sentinel/outputs/reports/

📊 MODÜL 3 ÖZET:
   Analiz edilen hat     : 870
   İptal oranı (ort)     : %10.2
   Graf düğüm sayısı     : 752
   Graf kenar sayısı     : 3,166
   En kritik hat         : AVR1
   Max domino etkisi     : 183 hat

🎯 Modül 3 tamamlandı! → Modül 5 (Etkinlik Anomalisi)


In [30]:
# ============================================================
# İETT SENTINEL — HÜCRE 10   MODÜL 5: ETKİNLİK ANOMALİSİ
# "Normal günlere göre anormal yoğunluk yaratan günleri tespit et"
# ============================================================

import gc
import warnings
warnings.filterwarnings('ignore')
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

print("=" * 60)
print("🎉 MODÜL 5: ETKİNLİK BAZLI ANOMALİ TESPİTİ")
print("=" * 60)

SAVE = '/content/drive/MyDrive/iett_sentinel/outputs/reports/'

# ── ADIM 1: Günlük Sefer Yoğunluğu ──────────────────────
print("\n📅 ADIM 1: Günlük sefer yoğunluğu hesaplanıyor...")

f_sefer_c = f_sefer.copy()
f_sefer_c['BASLANGICZAMANI'] = pd.to_datetime(
    f_sefer_c['BASLANGICZAMANI'], errors='coerce')

# ── FIX 1: 1969/2008 sahte tarihleri filtrele ────────────
f_sefer_c = f_sefer_c[
    f_sefer_c['BASLANGICZAMANI'].dt.year.between(2024, 2026)
].copy()
print(f"  ✅ Geçerli tarih sonrası: {len(f_sefer_c):,} sefer")

f_sefer_c['TARIH'] = f_sefer_c['BASLANGICZAMANI'].dt.date
f_sefer_c['YIL']   = f_sefer_c['BASLANGICZAMANI'].dt.year
f_sefer_c['AY']    = f_sefer_c['BASLANGICZAMANI'].dt.month
f_sefer_c['GUN']   = f_sefer_c['BASLANGICZAMANI'].dt.day
f_sefer_c['HGUN']  = f_sefer_c['BASLANGICZAMANI'].dt.dayofweek

# ── FIX 2: D_TARIH 2000'den başlıyor, 2024-2025 yok
# → Türkiye resmi tatillerini manuel tanımla
resmi_tatiller = [
    # 2024
    '2024-01-01',  # Yılbaşı
    '2024-04-10','2024-04-11','2024-04-12','2024-04-13',  # Ramazan Bayramı
    '2024-04-23',  # Ulusal Egemenlik
    '2024-05-01',  # İşçi Bayramı
    '2024-05-19',  # Atatürk'ü Anma
    '2024-06-16','2024-06-17','2024-06-18','2024-06-19',  # Kurban Bayramı
    '2024-07-15',  # Demokrasi Şehitleri
    '2024-08-30',  # Zafer Bayramı
    '2024-10-29',  # Cumhuriyet Bayramı
    # 2025
    '2025-01-01',  # Yılbaşı
    '2025-03-30','2025-03-31','2025-04-01','2025-04-02',  # Ramazan Bayramı
    '2025-04-23',  # Ulusal Egemenlik
    '2025-05-01',  # İşçi Bayramı
    '2025-05-19',  # Atatürk'ü Anma
    '2025-06-06','2025-06-07','2025-06-08','2025-06-09',  # Kurban Bayramı
    '2025-07-15',  # Demokrasi Şehitleri
    '2025-08-30',  # Zafer Bayramı
    '2025-10-29',  # Cumhuriyet Bayramı
]
resmi_tatil_set = set(pd.to_datetime(resmi_tatiller).date)

f_sefer_c['HAFTASONU']  = (f_sefer_c['HGUN'] >= 5).astype(int)
f_sefer_c['RESMITATIL'] = f_sefer_c['TARIH'].apply(
    lambda x: 1 if x in resmi_tatil_set else 0)

print(f"  ✅ Resmi tatil günü (manuel): {f_sefer_c['RESMITATIL'].sum():,} kayıt")
print(f"  ✅ Haftasonu kayıt          : {f_sefer_c['HAFTASONU'].sum():,} kayıt")

# Günlük sefer sayısı
gunluk_sefer = (f_sefer_c
    .groupby('TARIH')
    .agg(
        sefer_sayisi  =('FID',         'count'),
        benzersiz_hat =('SIDHAT',      'nunique'),
        haftasonu     =('HAFTASONU',   'max'),
        resmi_tatil   =('RESMITATIL',  'max'),
        yil           =('YIL',         'first'),
        ay            =('AY',          'first'),
        hgun          =('HGUN',        'first'),
    )
    .reset_index()
)
gunluk_sefer['TARIH'] = pd.to_datetime(gunluk_sefer['TARIH'])
gunluk_sefer = gunluk_sefer.sort_values('TARIH')

print(f"  ✅ Günlük kayıt      : {len(gunluk_sefer):,} gün")
print(f"  ✅ Tarih aralığı     : {gunluk_sefer['TARIH'].min().date()} → "
      f"{gunluk_sefer['TARIH'].max().date()}")
print(f"  ✅ Sefer aralığı     : {gunluk_sefer['sefer_sayisi'].min():,} — "
      f"{gunluk_sefer['sefer_sayisi'].max():,}")
print(f"  ✅ Ortalama sefer/gün: {gunluk_sefer['sefer_sayisi'].mean():,.0f}")
print(f"  ✅ Resmi tatil günü  : {gunluk_sefer['resmi_tatil'].sum():,}")

# ── ADIM 2: Günlük Arıza Yoğunluğu ──────────────────────
print("\n🔧 ADIM 2: Günlük arıza yoğunluğu...")

f_ariza_temiz['TARIH_DATE'] = f_ariza_temiz['OLAYTARIHI'].dt.date

gunluk_ariza = (f_ariza_temiz
    .groupby('TARIH_DATE')
    .agg(
        ariza_sayisi =('FID',            'count'),
        zayi_sefer   =('ZAYISEFERSAYISI','sum'),
    )
    .reset_index()
)
gunluk_ariza['TARIH'] = pd.to_datetime(gunluk_ariza['TARIH_DATE'])

gunluk = gunluk_sefer.merge(
    gunluk_ariza[['TARIH','ariza_sayisi','zayi_sefer']],
    on='TARIH', how='left'
).fillna({'ariza_sayisi':0, 'zayi_sefer':0})

print(f"  ✅ Birleşik günlük tablo: {len(gunluk):,} gün")

# ── ADIM 3: Anomali Tespiti (Z-Score) ───────────────────
print("\n🔍 ADIM 3: Anomali tespiti (Z-Score)...")

gunluk['HGUN_TIP'] = gunluk['hgun'].apply(
    lambda x: 'haftasonu' if x >= 5 else 'haftaici')

for tip in ['haftaici','haftasonu']:
    mask = gunluk['HGUN_TIP'] == tip
    mu   = gunluk.loc[mask, 'sefer_sayisi'].mean()
    std  = gunluk.loc[mask, 'sefer_sayisi'].std()
    if std > 0:
        gunluk.loc[mask, 'SEFER_ZSCORE'] = (
            (gunluk.loc[mask, 'sefer_sayisi'] - mu) / std
        ).round(3)

mu_a  = gunluk['ariza_sayisi'].mean()
std_a = gunluk['ariza_sayisi'].std()
gunluk['ARIZA_ZSCORE'] = ((gunluk['ariza_sayisi'] - mu_a) / std_a).round(3)

esik = 2.0
gunluk['SEFER_ANOMALI'] = gunluk['SEFER_ZSCORE'].abs() > esik
gunluk['ARIZA_ANOMALI'] = gunluk['ARIZA_ZSCORE'].abs() > esik
gunluk['CIFT_ANOMALI']  = gunluk['SEFER_ANOMALI'] & gunluk['ARIZA_ANOMALI']

print(f"  ✅ Sefer anomali günü  : {gunluk['SEFER_ANOMALI'].sum():,}")
print(f"  ✅ Arıza anomali günü  : {gunluk['ARIZA_ANOMALI'].sum():,}")
print(f"  ✅ Çift anomali günü   : {gunluk['CIFT_ANOMALI'].sum():,}")

# ── ADIM 4: Etkinlik Takvimi ─────────────────────────────
print("\n🎯 ADIM 4: Etkinlik takvimi ile eşleştirme...")

etkinlik_adaylari = gunluk[
    (gunluk['SEFER_ANOMALI']) |
    (gunluk['resmi_tatil'] == 1)
].copy()

etkinlik_adaylari['ETKİNLİK_TAHMINI'] = etkinlik_adaylari.apply(
    lambda r: (
        'Resmi Tatil + Yüksek Sefer' if r['resmi_tatil']==1 and r['SEFER_ZSCORE']>esik
        else 'Resmi Tatil + Düşük Sefer' if r['resmi_tatil']==1 and r['SEFER_ZSCORE']<-esik
        else 'Anormal Yüksek Sefer'      if r['SEFER_ZSCORE'] >  esik
        else 'Anormal Düşük Sefer'       if r['SEFER_ZSCORE'] < -esik
        else 'Resmi Tatil (Normal Sefer)'
    ), axis=1
)

print(f"\n  Etkinlik/Anomali dağılımı:")
print(etkinlik_adaylari['ETKİNLİK_TAHMINI'].value_counts().to_string())
print(f"\n  Resmi tatil günleri:")
print(etkinlik_adaylari[etkinlik_adaylari['resmi_tatil']==1]
      [['TARIH','sefer_sayisi','SEFER_ZSCORE','ETKİNLİK_TAHMINI']]
      .to_string(index=False))

# ── ADIM 5: Mevsimsel Örüntü ─────────────────────────────
print("\n📆 ADIM 5: Mevsimsel örüntü...")

aylik_ozet = (gunluk
    .groupby(['yil','ay'])
    .agg(
        ort_sefer   =('sefer_sayisi', 'mean'),
        ort_ariza   =('ariza_sayisi', 'mean'),
        anomali_gun =('SEFER_ANOMALI','sum'),
        toplam_gun  =('TARIH',        'count'),
    )
    .reset_index()
)
aylik_ozet['ANOMALI_ORAN'] = (
    aylik_ozet['anomali_gun'] / aylik_ozet['toplam_gun'] * 100
).round(1)
ay_adi = {1:'Oca',2:'Şub',3:'Mar',4:'Nis',5:'May',6:'Haz',
          7:'Tem',8:'Ağu',9:'Eyl',10:'Eki',11:'Kas',12:'Ara'}
aylik_ozet['ay_adi'] = aylik_ozet['yil'].astype(str) + '-' + aylik_ozet['ay'].map(ay_adi)

print(aylik_ozet[['ay_adi','ort_sefer','ort_ariza',
                   'anomali_gun','ANOMALI_ORAN']].to_string(index=False))

# ── ADIM 6: Haftanın Günü Profili ────────────────────────
print("\n📊 ADIM 6: Haftanın günü profili...")

gun_adi = {0:'Pzt',1:'Sal',2:'Çar',3:'Per',4:'Cum',5:'Cmt',6:'Paz'}
gunluk['gun_adi'] = gunluk['hgun'].map(gun_adi)

hgun_ozet = (gunluk
    .groupby('gun_adi')
    .agg(
        ort_sefer =('sefer_sayisi','mean'),
        ort_ariza =('ariza_sayisi','mean'),
        max_sefer =('sefer_sayisi','max'),
    )
    .reset_index()
)
hgun_ozet['sira'] = hgun_ozet['gun_adi'].map({v:k for k,v in gun_adi.items()})
hgun_ozet = hgun_ozet.sort_values('sira')
print(hgun_ozet[['gun_adi','ort_sefer','ort_ariza','max_sefer']].round(0).to_string(index=False))

# ── ADIM 7: En Anomalili Günler ──────────────────────────
print("\n🚨 ADIM 7: En anomalili günler (Top 20)...")

top_anomali = (gunluk
    .sort_values('SEFER_ZSCORE', ascending=False)
    .head(20)
    [['TARIH','sefer_sayisi','ariza_sayisi','SEFER_ZSCORE',
      'ARIZA_ZSCORE','haftasonu','resmi_tatil']]
)
print(top_anomali.to_string(index=False))

print(f"\n  En düşük sefer günleri:")
bot_anomali = (gunluk
    .sort_values('SEFER_ZSCORE')
    .head(10)
    [['TARIH','sefer_sayisi','SEFER_ZSCORE','resmi_tatil','haftasonu']]
)
print(bot_anomali.to_string(index=False))

# ── ADIM 8: Hat Bazında Anomali ──────────────────────────
print("\n🚌 ADIM 8: Hat bazında yoğunluk anomalisi...")

gunluk_hat = (f_sefer_c
    .groupby(['TARIH','SIDHAT'])
    .size()
    .reset_index(name='sefer_sayisi')
)

hat_anomali_list = []
for hat_id, grp in gunluk_hat.groupby('SIDHAT'):
    if len(grp) < 10:
        continue
    mu_h  = grp['sefer_sayisi'].mean()
    std_h = grp['sefer_sayisi'].std()
    if std_h == 0:
        continue
    grp = grp.copy()
    grp['Z'] = (grp['sefer_sayisi'] - mu_h) / std_h
    anomaliler = grp[grp['Z'].abs() > 2.5]
    if len(anomaliler) > 0:
        hat_anomali_list.append({
            'SIDHAT'     : hat_id,
            'anomali_gun': len(anomaliler),
            'max_z'      : grp['Z'].max(),
            'ort_sefer'  : mu_h,
        })

hat_anomali_df = pd.DataFrame(hat_anomali_list)
if len(hat_anomali_df) > 0:
    hat_anomali_df = hat_anomali_df.merge(
        d_hat_temiz[['SID','HATKODU','HATILCE']],
        left_on='SIDHAT', right_on='SID', how='left'
    ).sort_values('anomali_gun', ascending=False)
    print(f"\n  En çok anomali yaşayan 15 hat:")
    print(hat_anomali_df[['HATKODU','HATILCE','anomali_gun','max_z','ort_sefer']]
          .head(15).round(2).to_string(index=False))

# ── ADIM 9: Görselleştirmeler ─────────────────────────────
print("\n📊 ADIM 9: Görselleştirmeler...")

# Grafik 1: Zaman serisi
fig1 = go.Figure()
fig1.add_trace(go.Scatter(
    x=gunluk['TARIH'], y=gunluk['sefer_sayisi'],
    mode='lines', name='Günlük Sefer',
    line=dict(color='#3498db', width=1)
))
an = gunluk[gunluk['SEFER_ANOMALI']]
fig1.add_trace(go.Scatter(
    x=an['TARIH'], y=an['sefer_sayisi'],
    mode='markers', name='Anomali',
    marker=dict(color='#e74c3c', size=8, symbol='star')
))
tat = gunluk[gunluk['resmi_tatil']==1]
if len(tat) > 0:
    fig1.add_trace(go.Scatter(
        x=tat['TARIH'], y=tat['sefer_sayisi'],
        mode='markers', name='Resmi Tatil',
        marker=dict(color='#f39c12', size=8, symbol='diamond')
    ))
fig1.update_layout(
    title='📅 Günlük Sefer Sayısı — Anomaliler & Tatiller',
    template='plotly_dark', width=1000, height=480
)
fig1.show()

# Grafik 2: Aylık anomali
fig2 = px.bar(
    aylik_ozet, x='ay_adi', y='ANOMALI_ORAN',
    color='ort_ariza', color_continuous_scale='RdYlGn_r',
    title='📆 Aylık Anomali Oranı (%) & Ortalama Arıza',
    template='plotly_dark', width=850, height=430,
    labels={'ay_adi':'Ay','ANOMALI_ORAN':'Anomali Oranı (%)'}
)
fig2.show()

# Grafik 3: Haftanın günü
fig3 = px.bar(
    hgun_ozet, x='gun_adi', y='ort_sefer',
    color='ort_ariza', color_continuous_scale='Reds',
    title='📊 Haftanın Günü × Ortalama Sefer & Arıza',
    template='plotly_dark', width=750, height=430,
    labels={'gun_adi':'Gün','ort_sefer':'Ort. Sefer'}
)
fig3.show()

# Grafik 4: Z-Score dağılımı
fig4 = px.histogram(
    gunluk, x='SEFER_ZSCORE', nbins=50,
    color='HGUN_TIP',
    color_discrete_map={'haftaici':'#3498db','haftasonu':'#e74c3c'},
    title='📊 Günlük Sefer Z-Score Dağılımı',
    template='plotly_dark', width=800, height=420,
    labels={'SEFER_ZSCORE':'Z-Score'}
)
fig4.add_vline(x=2,  line_dash='dash', line_color='yellow', annotation_text='+2σ')
fig4.add_vline(x=-2, line_dash='dash', line_color='yellow', annotation_text='-2σ')
fig4.show()

# Grafik 5: Hat anomali
if len(hat_anomali_df) > 0:
    fig5 = px.bar(
        hat_anomali_df.head(15).sort_values('anomali_gun'),
        x='anomali_gun', y='HATKODU', orientation='h',
        color='max_z', color_continuous_scale='RdYlGn_r',
        title='🚌 Hat Bazında Anomali Günü Sayısı (Top 15)',
        template='plotly_dark', width=850, height=500,
        labels={'HATKODU':'Hat','anomali_gun':'Anomali Günü'}
    )
    fig5.show()

# ── Kaydet ───────────────────────────────────────────────
gunluk.to_csv(SAVE + 'modul5_gunluk_anomali.csv', index=False)
etkinlik_adaylari.to_csv(SAVE + 'modul5_etkinlik_adaylari.csv', index=False)
if len(hat_anomali_df) > 0:
    hat_anomali_df.to_csv(SAVE + 'modul5_hat_anomali.csv', index=False)

gc.collect()
print(f"\n✅ Sonuçlar kaydedildi → {SAVE}")
print(f"\n📊 MODÜL 5 ÖZET:")
print(f"   Analiz edilen gün      : {len(gunluk):,}")
print(f"   Tarih aralığı          : {gunluk['TARIH'].min().date()} → {gunluk['TARIH'].max().date()}")
print(f"   Sefer anomali günü     : {gunluk['SEFER_ANOMALI'].sum():,}")
print(f"   Arıza anomali günü     : {gunluk['ARIZA_ANOMALI'].sum():,}")
print(f"   Çift anomali günü      : {gunluk['CIFT_ANOMALI'].sum():,}")
print(f"   Resmi tatil günü       : {gunluk['resmi_tatil'].sum():,}")
print(f"   Etkinlik adayı gün     : {len(etkinlik_adaylari):,}")
if len(hat_anomali_df) > 0:
    print(f"   En anomalili hat       : "
          f"{hat_anomali_df.iloc[0]['HATKODU']} "
          f"({hat_anomali_df.iloc[0]['anomali_gun']} gün)")
print(f"\n🎯 Modül 5 tamamlandı! → Hücre 11 (SENTINEL Dashboard)")

🎉 MODÜL 5: ETKİNLİK BAZLI ANOMALİ TESPİTİ

📅 ADIM 1: Günlük sefer yoğunluğu hesaplanıyor...
  ✅ Geçerli tarih sonrası: 12,143,277 sefer
  ✅ Resmi tatil günü (manuel): 583,126 kayıt
  ✅ Haftasonu kayıt          : 2,907,998 kayıt
  ✅ Günlük kayıt      : 275 gün
  ✅ Tarih aralığı     : 2024-01-01 → 2025-04-02
  ✅ Sefer aralığı     : 1 — 50,425
  ✅ Ortalama sefer/gün: 44,157
  ✅ Resmi tatil günü  : 17

🔧 ADIM 2: Günlük arıza yoğunluğu...
  ✅ Birleşik günlük tablo: 275 gün

🔍 ADIM 3: Anomali tespiti (Z-Score)...
  ✅ Sefer anomali günü  : 3
  ✅ Arıza anomali günü  : 8
  ✅ Çift anomali günü   : 0

🎯 ADIM 4: Etkinlik takvimi ile eşleştirme...

  Etkinlik/Anomali dağılımı:
ETKİNLİK_TAHMINI
Resmi Tatil (Normal Sefer)    15
Resmi Tatil + Düşük Sefer      2
Anormal Düşük Sefer            1

  Resmi tatil günleri:
     TARIH  sefer_sayisi  SEFER_ZSCORE           ETKİNLİK_TAHMINI
2024-01-01         38306         -1.35 Resmi Tatil (Normal Sefer)
2024-04-10         40555         -0.99 Resmi Tatil (Nor


✅ Sonuçlar kaydedildi → /content/drive/MyDrive/iett_sentinel/outputs/reports/

📊 MODÜL 5 ÖZET:
   Analiz edilen gün      : 275
   Tarih aralığı          : 2024-01-01 → 2025-04-02
   Sefer anomali günü     : 3
   Arıza anomali günü     : 8
   Çift anomali günü      : 0
   Resmi tatil günü       : 17
   Etkinlik adayı gün     : 18
   En anomalili hat       : H-1 (23 gün)

🎯 Modül 5 tamamlandı! → Hücre 11 (SENTINEL Dashboard)


In [32]:
# ============================================================
# İETT SENTINEL — HÜCRE 11: SENTINEL DASHBOARD
# Tüm modüllerin çıktısını bir arada gösteren panel
# ============================================================

import gc
import warnings
warnings.filterwarnings('ignore')
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import pandas as pd
import numpy as np

print("=" * 60)
print("🚨 İETT SENTINEL DASHBOARD")
print("=" * 60)

SAVE = '/content/drive/MyDrive/iett_sentinel/outputs/reports/'

# ── PANEL 1: GENEL DURUM KARTLARı ────────────────────────
print("\n📊 PANEL 1: Genel Durum Özeti...")

# KPI verileri — tüm modüllerden topla
kpi_data = {
    'Toplam Arıza'       : f_ariza_temiz['FID'].count(),
    'Yüksek Riskli Araç' : (arac_features['RİSK_SEVİYESİ']=='🔴 YÜKSEK RİSK').sum(),
    'Toplam Zayi Sefer'  : int(arac_features['toplam_zayi_sefer'].sum()),
    'Kriz Tetikleyici'   : int(f_kaza_temiz['KRİZ_TETIKLEYICI'].sum()),
    'Kritik Hat'         : merkezi_df.iloc[0]['hat_adi'],
    'Max Domino Etkisi'  : f"{sim_df['etkilenen_hat'].max()} hat",
    'Denetim Korelasyon' : f"r={korel:.3f}",
    'Olumsuz Denetim'    : f"%{denetim_ozet['OLUMSUZ_ORAN'].mean()*100:.1f}",
}

print("\n  📌 KPI Özeti:")
for k, v in kpi_data.items():
    print(f"     {k:<25}: {v:>15}")

# ── PANEL 2: ANA DASHBOARD ───────────────────────────────
print("\n📊 PANEL 2: Ana Dashboard oluşturuluyor...")

fig = make_subplots(
    rows=4, cols=3,
    subplot_titles=[
        '🔴 Risk Dağılımı',
        '📈 Aylık Arıza Trend',
        '🔗 Birliktelik — Top 10 Kural',
        '⏱️ Denetim→Arıza Gecikme',
        '🕸️ Domino Etkisi',
        '🌤️ Hava × Kaza',
        '📅 Günlük Sefer Anomalisi',
        '🚌 Hat İptal Oranı (Top 10)',
        '📊 Araç Yaşı × Risk',
        '🗺️ İlçe Kaza Riski',
        '🔧 Arıza Türü × Zayi Sefer',
        '📆 Haftanın Günü Profili',
    ],
    specs=[
        [{'type':'domain'}, {'type':'xy'}, {'type':'xy'}],
        [{'type':'xy'},     {'type':'xy'}, {'type':'xy'}],
        [{'type':'xy'},     {'type':'xy'}, {'type':'xy'}],
        [{'type':'xy'},     {'type':'xy'}, {'type':'xy'}],
    ],
    vertical_spacing=0.08,
    horizontal_spacing=0.08,
)

renk_map = {
    '🔴 YÜKSEK RİSK' : '#e74c3c',
    '🟠 ORTA-YÜKSEK' : '#e67e22',
    '🟡 ORTA RİSK'   : '#f1c40f',
    '🟢 DÜŞÜK RİSK'  : '#2ecc71',
    '🔵 ÇOK DÜŞÜK'   : '#3498db',
    '⚪ MİNİMAL'      : '#95a5a6',
}

# ── Grafik 1: Risk Dağılımı Pie ──────────────────────────
risk_dag = arac_features['RİSK_SEVİYESİ'].value_counts()
fig.add_trace(go.Pie(
    labels=risk_dag.index,
    values=risk_dag.values,
    marker_colors=[renk_map.get(r,'#aaa') for r in risk_dag.index],
    hole=0.4,
    textinfo='label+percent',
    textfont_size=9,
    showlegend=False,
), row=1, col=1)

# ── Grafik 2: Aylık Arıza Trend ──────────────────────────
for yil, renk in [(2024,'#3498db'),(2025,'#e74c3c')]:
    d = aylik_ariza[aylik_ariza['YIL'] == yil]
    fig.add_trace(go.Scatter(
        x=d['TARIH'], y=d['ariza_sayisi'],
        name=f'Arıza {yil}', line=dict(color=renk, width=2),
        mode='lines+markers', marker=dict(size=4),
        showlegend=True,
    ), row=1, col=2)

# ── Grafik 3: Birliktelik Kuralları ──────────────────────
if len(rules_df) > 0:
    top_rules = rules_df.head(10).copy()
    top_rules['kural'] = (top_rules['oncul'].str[:15] + ' → ' +
                          top_rules['sonuc'].str[:15])
    fig.add_trace(go.Bar(
        x=top_rules['confidence'],
        y=top_rules['kural'],
        orientation='h',
        marker_color='#9b59b6',
        showlegend=False,
        text=top_rules['lift'].round(2),
        textposition='outside',
    ), row=1, col=3)

# ── Grafik 4: Denetim→Arıza Gecikme ─────────────────────
if len(ariza_pozitif) > 0:
    gec_hist = ariza_pozitif[
        ariza_pozitif['GECIKME_GUN'] <= 60
    ]['GECIKME_GUN'].value_counts().sort_index()
    fig.add_trace(go.Bar(
        x=gec_hist.index, y=gec_hist.values,
        marker_color='#e74c3c', showlegend=False,
    ), row=2, col=1)
    # vline → shape olarak ekle (Pie chart çakışmasını önler)
    fig.add_shape(
        type='line', x0=14, x1=14, y0=0, y1=1,
        yref='paper', xref='x4',
        line=dict(color='yellow', dash='dash', width=1.5),
    )

# ── Grafik 5: Domino Etkisi ───────────────────────────────
if len(sim_df) > 0:
    fig.add_trace(go.Bar(
        x=sim_df['hat_adi'],
        y=sim_df['etkilenen_hat'],
        marker_color=['#e74c3c','#e67e22','#f1c40f','#3498db','#2ecc71'],
        showlegend=False,
        text=sim_df['etkilenen_hat'],
        textposition='outside',
    ), row=2, col=2)

# ── Grafik 6: Hava × Kaza ────────────────────────────────
hava_f = hava_kaza[hava_kaza['HAVA_DURUMU'] != 'BELİRSİZ'].copy()
fig.add_trace(go.Bar(
    x=hava_f['HAVA_DURUMU'],
    y=hava_f['kaza_sayisi'],
    marker_color='#3498db', showlegend=False,
    text=hava_f['ciddi_kaza'],
    textposition='outside',
), row=2, col=3)

# ── Grafik 7: Günlük Sefer Anomalisi ─────────────────────
fig.add_trace(go.Scatter(
    x=gunluk['TARIH'], y=gunluk['sefer_sayisi'],
    mode='lines', name='Günlük Sefer',
    line=dict(color='#3498db', width=1),
    showlegend=False,
), row=3, col=1)
an = gunluk[gunluk['SEFER_ANOMALI']]
fig.add_trace(go.Scatter(
    x=an['TARIH'], y=an['sefer_sayisi'],
    mode='markers', name='Anomali',
    marker=dict(color='#e74c3c', size=8, symbol='star'),
    showlegend=False,
), row=3, col=1)
tat = gunluk[gunluk['resmi_tatil']==1]
if len(tat) > 0:
    fig.add_trace(go.Scatter(
        x=tat['TARIH'], y=tat['sefer_sayisi'],
        mode='markers', name='Tatil',
        marker=dict(color='#f39c12', size=6, symbol='diamond'),
        showlegend=False,
    ), row=3, col=1)

# ── Grafik 8: Hat İptal Oranı ────────────────────────────
top_iptal = (hat_iptal[hat_iptal['toplam_sefer'] >= 100]
             .sort_values('IPTAL_ORANI', ascending=False)
             .head(10))
fig.add_trace(go.Bar(
    x=top_iptal['HATKODU'],
    y=top_iptal['IPTAL_ORANI'],
    marker_color='#e74c3c', showlegend=False,
), row=3, col=2)

# ── Grafik 9: Araç Yaşı × Risk ───────────────────────────
yas_risk = (arac_features
    .groupby(['yas_kategori','RİSK_SEVİYESİ'])
    .size().reset_index(name='sayi'))
for risk, renk in renk_map.items():
    d = yas_risk[yas_risk['RİSK_SEVİYESİ']==risk]
    if len(d) > 0:
        fig.add_trace(go.Bar(
            name=risk, x=d['yas_kategori'], y=d['sayi'],
            marker_color=renk, showlegend=False,
        ), row=3, col=3)

# ── Grafik 10: İlçe Kaza Riski ───────────────────────────
ilce_top = (ilce_kaza[ilce_kaza['ILCE'] != 'BELİRSİZ']
            .head(10))
fig.add_trace(go.Bar(
    x=ilce_top['ILCE'],
    y=ilce_top['kaza_sayisi'],
    marker_color=ilce_top['kriz_orani'],
    marker_colorscale='RdYlGn_r',
    showlegend=False,
), row=4, col=1)

# ── Grafik 11: Arıza Türü × Zayi Sefer ──────────────────
zayi = (f_ariza_temiz[f_ariza_temiz['ZAYISEFERSAYISI']>0]
        .groupby('ARIZAUSTKODTANIM')['ZAYISEFERSAYISI']
        .sum().sort_values(ascending=False).head(8).reset_index())
fig.add_trace(go.Bar(
    x=zayi['ZAYISEFERSAYISI'],
    y=zayi['ARIZAUSTKODTANIM'],
    orientation='h',
    marker_color='#e67e22', showlegend=False,
), row=4, col=2)

# ── Grafik 12: Haftanın Günü Profili ─────────────────────
fig.add_trace(go.Bar(
    x=hgun_ozet['gun_adi'],
    y=hgun_ozet['ort_sefer'],
    marker_color='#2ecc71', showlegend=False,
    name='Sefer',
), row=4, col=3)
fig.add_trace(go.Scatter(
    x=hgun_ozet['gun_adi'],
    y=hgun_ozet['ort_ariza'],
    mode='lines+markers',
    line=dict(color='#e74c3c', width=2),
    yaxis='y2', showlegend=False,
    name='Arıza',
), row=4, col=3)

# ── Layout ───────────────────────────────────────────────
fig.update_layout(
    title=dict(
        text='🚨 İETT SENTINEL — Akıllı Kriz Öngörü & Operasyonel Dayanıklılık Sistemi',
        font=dict(size=16, color='white'),
        x=0.5,
    ),
    template='plotly_dark',
    height=1800,
    width=1400,
    paper_bgcolor='#0d1117',
    plot_bgcolor='#161b22',
    font=dict(color='white', size=9),
    barmode='stack',
)

fig.update_xaxes(tickangle=45, tickfont=dict(size=8))
fig.update_yaxes(tickfont=dict(size=8))
fig.show()

# ── PANEL 3: KPI Kartları ─────────────────────────────────
print("\n📊 PANEL 3: KPI Özet Tablosu...")

kpi_fig = go.Figure()
kpi_fig.add_trace(go.Table(
    header=dict(
        values=['<b>Metrik</b>','<b>Değer</b>','<b>Yorum</b>'],
        fill_color='#e74c3c',
        font=dict(color='white', size=12),
        align='center',
        height=35,
    ),
    cells=dict(
        values=[
            ['Toplam Arıza Kaydı','Yüksek Riskli Araç',
             'Toplam Zayi Sefer','Kriz Tetikleyici Kaza',
             'En Kritik Hat','Max Domino Etkisi',
             'Denetim→Arıza Korelasyon','Ort. Olumsuz Denetim',
             'Denetim Sonrası 14 Gün Arıza','Kaza Pik Saati',
             'Arıza Pik Saati','En Riskli İlçe',
             'İptal Oranı (Sistem)','2024→2025 Arıza'],
            [f"{kpi_data['Toplam Arıza']:,}",
             f"{kpi_data['Yüksek Riskli Araç']:,}",
             f"{kpi_data['Toplam Zayi Sefer']:,}",
             f"{kpi_data['Kriz Tetikleyici']:,}",
             kpi_data['Kritik Hat'],
             kpi_data['Max Domino Etkisi'],
             kpi_data['Denetim Korelasyon'],
             kpi_data['Olumsuz Denetim'],
             '%71.3',
             f"{pik_kaza:02d}:00",
             f"{pik_ariza:02d}:00",
             ilce_kaza[ilce_kaza['ILCE']!='BELİRSİZ'].iloc[0]['ILCE'],
             '%10.2',
             '📈 %5.8'],
            ['2024-2025 arıza kayıtları',
             'Acil bakım öncelikli',
             'Operasyonel kayıp',
             'Müdahale gerektiren',
             'Sistem omurgası',
             'Aksaklık yayılımı',
             'Güçlü pozitif ilişki',
             'Kapsam genişletilmeli',
             '14 gün erken uyarı penceresi',
             'Yoğun saat müdahalesi',
             'Yoğun saat müdahalesi',
             'Öncelikli bölge',
             'Normal seviye',
             'Artan trend — dikkat'],
        ],
        fill_color=[['#161b22']*14],
        font=dict(color='white', size=11),
        align=['left','center','left'],
        height=30,
    )
))

kpi_fig.update_layout(
    title='📌 İETT SENTINEL — KPI Özet Tablosu',
    template='plotly_dark',
    height=550,
    paper_bgcolor='#0d1117',
)
kpi_fig.show()

# ── Kaydet ───────────────────────────────────────────────
fig.write_html(SAVE + '../figures/sentinel_dashboard.html')
kpi_fig.write_html(SAVE + '../figures/sentinel_kpi.html')

gc.collect()
print(f"\n✅ Dashboard kaydedildi → outputs/figures/")
print(f"\n{'='*60}")
print(f"🚨 İETT SENTINEL DASHBOARD TAMAMLANDI")
print(f"{'='*60}")
print(f"""
  📦 MODÜL ÖZETİ:
  ├── Modül 1 (Risk Kümeleme)    : {len(arac_features):,} araç analiz edildi
  ├── Modül 2 (Birliktelik)      : {len(rules_df)} kural, lift max {rules_df['lift'].max():.2f}
  ├── Modül 2B (Denetim-Arıza)  : r={korel:.3f}, %71.3 → 14 gün içi arıza
  ├── Modül 3 (Domino)           : {G.number_of_nodes()} hat, max {sim_df['etkilenen_hat'].max()} hat etkileniyor
  ├── Modül 4 (Kriz Zaman Serisi): pik 16:00, kriz tetikleyici {f_kaza_temiz['KRİZ_TETIKLEYICI'].sum()}
  └── Modül 5 (Etkinlik Anomali) : {len(gunluk)} gün, {gunluk['SEFER_ANOMALI'].sum()} anomali günü

  🎯 SENTINEL'in 3 Temel Çıktısı:
  1. BAKIM ÖNCELİK LİSTESİ → {(arac_features['RİSK_SEVİYESİ']=='🔴 YÜKSEK RİSK').sum()} araç acil bakım bekliyor
  2. ERKEN UYARI             → Olumsuz denetimden 14 gün içinde %71.3 arıza
  3. KRİZ YÖNETİMİ          → AVR1 kapanırsa 183 hat etkilenir
""")

🚨 İETT SENTINEL DASHBOARD

📊 PANEL 1: Genel Durum Özeti...

  📌 KPI Özeti:
     Toplam Arıza             :          333386
     Yüksek Riskli Araç       :            3120
     Toplam Zayi Sefer        :          206960
     Kriz Tetikleyici         :             403
     Kritik Hat               :            AVR1
     Max Domino Etkisi        :         183 hat
     Denetim Korelasyon       :         r=0.545
     Olumsuz Denetim          :           %21.9

📊 PANEL 2: Ana Dashboard oluşturuluyor...



📊 PANEL 3: KPI Özet Tablosu...



✅ Dashboard kaydedildi → outputs/figures/

🚨 İETT SENTINEL DASHBOARD TAMAMLANDI

  📦 MODÜL ÖZETİ:
  ├── Modül 1 (Risk Kümeleme)    : 6,769 araç analiz edildi
  ├── Modül 2 (Birliktelik)      : 380 kural, lift max 1.49
  ├── Modül 2B (Denetim-Arıza)  : r=0.545, %71.3 → 14 gün içi arıza
  ├── Modül 3 (Domino)           : 752 hat, max 183 hat etkileniyor
  ├── Modül 4 (Kriz Zaman Serisi): pik 16:00, kriz tetikleyici 403
  └── Modül 5 (Etkinlik Anomali) : 275 gün, 3 anomali günü

  🎯 SENTINEL'in 3 Temel Çıktısı:
  1. BAKIM ÖNCELİK LİSTESİ → 3120 araç acil bakım bekliyor
  2. ERKEN UYARI             → Olumsuz denetimden 14 gün içinde %71.3 arıza
  3. KRİZ YÖNETİMİ          → AVR1 kapanırsa 183 hat etkilenir

